# PANINI course project — student-like reference solution and Colab run

This notebook presents one complete, reproducible solution to Questions
1–12. It audits both 100-question packages, constructs and analyzes the
alternative memory graphs, evaluates retrieval, implements connected-DAG
RICR, runs the neural system on all 200 questions, performs the required
ablations, and writes the four submission JSONL files. Expensive work is
appended to Drive after every question, so `Run all` can be repeated
after a Colab interruption without losing completed records.

In [ ]:
# Run controls. The defaults perform the complete assignment.
import os

# Keep the submitted notebook at full-run defaults.  The environment
# overrides are useful for a fast CPU-only structural validation.
RUN_FULL = os.environ.get('PANINI_RUN_FULL', '1') == '1'
RUN_ABLATIONS = os.environ.get('PANINI_RUN_ABLATIONS', '1') == '1'
MOUNT_DRIVE = True
DATASETS = ('2wiki', 'musique')

# For a quick validation before the full run, set this to slice(0, 2).
# Return it to slice(None) for the required 200-question run.
QUESTION_SLICE = slice(None)

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    REPO_ROOT = Path('/content/panini-course-project')
    if not (REPO_ROOT / 'manifest.json').exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/YigitTurali/panini-course-project.git',
            str(REPO_ROOT),
        ], check=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '-r',
        str(REPO_ROOT / 'requirements-colab.txt')
    ], check=True)
    sys.path.insert(0, str(REPO_ROOT))
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
        WORK_ROOT = Path('/content/drive/MyDrive/panini-full-answer-key')
    else:
        WORK_ROOT = Path('/content/panini-full-answer-key')
    PACKAGE_ROOTS = {
        '2wiki': REPO_ROOT,
        'musique': REPO_ROOT / 'packages/panini_musique_100',
    }
    AUDIT_CSV = None
else:
    source_override = os.environ.get('PANINI_SOURCE_ROOT')
    SOURCE_ROOT = Path(source_override).resolve() if source_override else Path.cwd().resolve()
    while SOURCE_ROOT != SOURCE_ROOT.parent and not (SOURCE_ROOT / 'course_project').exists():
        SOURCE_ROOT = SOURCE_ROOT.parent
    if not (SOURCE_ROOT / 'course_project').exists():
        raise FileNotFoundError('Set PANINI_SOURCE_ROOT to the gsw-memory checkout.')
    sys.path.insert(0, str(SOURCE_ROOT / 'course_project/src'))
    WORK_ROOT = Path(os.environ.get(
        'PANINI_WORK_ROOT',
        SOURCE_ROOT / 'course_project/instructor/full_run_v4'))
    PACKAGE_ROOTS = {
        '2wiki': SOURCE_ROOT / 'course_project/release/panini_2wiki_100',
        'musique': SOURCE_ROOT / 'course_project/release/panini_musique_100',
    }
    AUDIT_CSV = SOURCE_ROOT / 'course_project/instructor/reconciliation_audit.csv'

CACHE_BASE = WORK_ROOT / 'cache'
OUTPUT_ROOT = WORK_ROOT / 'submission'
CACHE_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print({'work_root': str(WORK_ROOT), 'colab': IN_COLAB})

## Solution implementation

The next cell is intentionally long and collapsed in Colab. It is
the complete reference implementation used by later answer cells:
conservative reconciliation, plan validation, global beam pruning,
exact-query caches, sequential model loading, metrics, and output
materialization. Keeping it inside the notebook makes this answer
key self-contained.

In [ ]:
"""Complete pipeline used by this Colab solution.

The functions in this file deliberately keep every expensive stage restartable.
The generated notebook embeds this module so it remains self-contained after it
is uploaded to Colab; this source copy exists so the implementation can be
tested and maintained normally.
"""

from __future__ import annotations

import gc
import json
import math
import os
import platform
import random
import re
import sys
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, replace
from itertools import product
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence


PLACEHOLDER = re.compile(r"<ENTITY_Q(\d+)>")
WORD = re.compile(r"\w+")
FORBIDDEN_HELDOUT_FIELDS = {
    "answer",
    "answer_aliases",
    "supporting_facts",
    "evidences",
}


@dataclass(frozen=True)
class RunConfig:
    seed: int = 232
    beam_width: int = 5
    candidates_per_hop: int = 15
    retrieval_pool: int = 60
    entity_top_k: int = 20
    qa_top_k: int = 60
    rrf_constant: float = 60.0
    retrieval_weight: float = 0.5
    rerank_batch_size: int = 2
    rerank_max_length: int = 512
    free_colab_rerank_batch_size: int = 1
    free_colab_rerank_max_length: int = 256
    reranker_model: str = "Qwen/Qwen3-Reranker-8B"
    free_colab_reranker_model: str = "Qwen/Qwen3-Reranker-4B"
    # The 8B reranker fits a 15 GiB T4 in 4-bit mode when loaded alone.
    reranker_8b_minimum_gib: float = 14.5
    multi_dependency_threshold: float = 0.3
    max_new_tokens: int = 768


def select_reranker_model(config: RunConfig, total_gib: float) -> str:
    """Select the 8B reference reranker or the T4-safe 4B fallback."""

    return (
        config.reranker_model
        if total_gib >= config.reranker_8b_minimum_gib
        else config.free_colab_reranker_model
    )


def read_jsonl(path: str | Path) -> list[dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    records: list[dict[str, Any]] = []
    with path.open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(
                    f"[cache warning] ignored incomplete JSONL line {line_number} in {path}",
                    flush=True,
                )
    return records


def append_jsonl(path: str | Path, record: Mapping[str, Any]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    needs_separator = path.exists() and path.stat().st_size > 0
    if needs_separator:
        with path.open("rb") as existing:
            existing.seek(-1, os.SEEK_END)
            needs_separator = existing.read(1) != b"\n"
    with path.open("a", encoding="utf-8") as handle:
        if needs_separator:
            handle.write("\n")
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def write_jsonl(path: str | Path, records: Iterable[Mapping[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def normalize_text(value: object) -> str:
    return " ".join(WORD.findall(str(value).casefold()))


def release_gpu() -> None:
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def gpu_snapshot() -> dict[str, Any]:
    try:
        import torch

        if not torch.cuda.is_available():
            return {"cuda": False}
        props = torch.cuda.get_device_properties(0)
        return {
            "cuda": True,
            "name": props.name,
            "total_gib": round(props.total_memory / 2**30, 3),
            "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3),
            "peak_allocated_gib": round(
                torch.cuda.max_memory_allocated() / 2**30, 3
            ),
        }
    except ImportError:
        return {"cuda": False}


def audit_package(package, dataset: str) -> tuple[list[dict], dict[str, Any]]:
    """Answer-key implementation for the Question 1 artifact audit."""

    import numpy as np

    public = package.questions("public")
    heldout = package.questions("held_out")
    entities = package.entities()
    qa_rows = package.qa_pairs()
    entity_ids = json.loads(
        (package.root / "embeddings/entity_ids.json").read_text()
    )
    qa_ids = json.loads((package.root / "embeddings/qa_ids.json").read_text())
    entity_matrix = np.load(
        package.root / "embeddings/entity_embeddings.npy", mmap_mode="r"
    )
    qa_matrix = np.load(
        package.root / "embeddings/qa_embeddings.npy", mmap_mode="r"
    )
    assert len(entity_ids) == len(set(entity_ids)) == entity_matrix.shape[0]
    assert len(qa_ids) == len(set(qa_ids)) == qa_matrix.shape[0]
    assert set(entity_ids) == {row["entity_uid"] for row in entities}
    assert set(qa_ids) == {row["qa_uid"] for row in qa_rows}
    assert all(
        not FORBIDDEN_HELDOUT_FIELDS.intersection(row) for row in heldout
    )
    document_ids = {row["document_id"] for row in package.documents()}
    gsw_paths = package.gsw_paths()
    validation_count = len(package.decompositions())
    rows = [
        {
            "dataset": dataset,
            "split": "development",
            "questions": len(public),
            "unique_documents": len(
                {doc for row in public for doc in row["context_document_ids"]}
            ),
            "gsw_files": len(gsw_paths),
            "entities": len(entities),
            "qa_records": len(qa_rows),
            "reviewed_decompositions": validation_count,
        },
        {
            "dataset": dataset,
            "split": "held_out",
            "questions": len(heldout),
            "unique_documents": len(
                {doc for row in heldout for doc in row["context_document_ids"]}
            ),
            "gsw_files": len(gsw_paths),
            "entities": len(entities),
            "qa_records": len(qa_rows),
            "reviewed_decompositions": validation_count,
        },
    ]
    examples = {
        "question": public[0],
        "entity": entities[0],
        "qa": qa_rows[0],
        "document_count": len(document_ids),
    }
    return rows, examples


STOPWORDS = {
    "a", "an", "and", "at", "by", "for", "from", "in", "is", "of",
    "on", "or", "the", "to", "was", "with",
}
GENERIC_ATTRIBUTE_ROLES = {
    "achievement", "classification", "date", "date range", "ethnicity",
    "frequency", "genre", "language", "nationality", "number", "occupation",
    "profession", "quantity", "time", "time period", "title", "year",
}


def _role_parts(attributes: Mapping[str, Any]) -> tuple[set[str], set[str]]:
    labels: set[str] = set()
    state_tokens: set[str] = set()
    for item in attributes.get("roles", []):
        if isinstance(item, Mapping):
            labels.add(normalize_text(item.get("role", "")))
            for state in item.get("states", []):
                state_tokens.update(WORD.findall(str(state).casefold()))
        else:
            labels.add(normalize_text(item))
    state_tokens.difference_update(STOPWORDS)
    return {value for value in labels if value}, state_tokens


def _neighbor_signature(native, node: str) -> set[str]:
    signatures: set[str] = set()
    for verb in native.predecessors(node):
        phrase = native.nodes[verb].get("phrase", "")
        signatures.update(WORD.findall(str(phrase).casefold()))
        for sibling in native.successors(verb):
            if sibling == node:
                continue
            name = native.nodes[sibling].get("name", "")
            signatures.update(WORD.findall(str(name).casefold()))
    signatures.difference_update(STOPWORDS)
    return signatures


def conservative_entity_mapping(native) -> tuple[dict[str, str], list[dict]]:
    """Return a conservative cross-document occurrence-to-identity map.

    Two occurrences are eligible only when canonical surfaces and node types
    match. Generic attribute values and numeric/date surfaces are not merged.
    A multi-token proper surface is joined when role labels overlap and neither
    occurrence is only a generic attribute; otherwise two content-bearing
    neighborhood tokens must overlap. Empty evidence never counts as
    agreement. Union-find makes transitive clusters, and every decision retains
    its evidence for the audit table.
    """

    from panini_course.graph import canonical_entity_name

    occurrences = [
        node
        for node, data in native.nodes(data=True)
        if data.get("node_type") != "verb_phrase"
    ]
    parent = {node: node for node in occurrences}

    def find(node: str) -> str:
        while parent[node] != node:
            parent[node] = parent[parent[node]]
            node = parent[node]
        return node

    def union(left: str, right: str) -> None:
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            parent[max(left_root, right_root)] = min(left_root, right_root)

    grouped: dict[tuple[str, str], list[str]] = defaultdict(list)
    for node in occurrences:
        data = native.nodes[node]
        key = (
            canonical_entity_name(str(data.get("name", ""))),
            str(data.get("node_type", "unknown")),
        )
        if key[0]:
            grouped[key].append(node)

    decisions: list[dict] = []
    for (surface, node_type), nodes in sorted(grouped.items()):
        if len(nodes) < 2:
            continue
        roles = {node: _role_parts(native.nodes[node]) for node in nodes}
        neighborhoods = {node: _neighbor_signature(native, node) for node in nodes}
        for index, left in enumerate(nodes):
            for right in nodes[index + 1 :]:
                if native.nodes[left].get("document_id") == native.nodes[right].get(
                    "document_id"
                ):
                    continue
                left_labels, left_states = roles[left]
                right_labels, right_states = roles[right]
                state_union = left_states | right_states
                state_jaccard = (
                    len(left_states & right_states) / len(state_union)
                    if left_states and right_states and state_union
                    else 0.0
                )
                neighborhood_overlap = sorted(
                    neighborhoods[left] & neighborhoods[right]
                )
                surface_tokens = surface.split()
                numeric_surface = not any(character.isalpha() for character in surface)
                generic = (
                    bool(left_labels) and left_labels <= GENERIC_ATTRIBUTE_ROLES
                ) or (
                    bool(right_labels) and right_labels <= GENERIC_ATTRIBUTE_ROLES
                )
                compatible_roles = bool(left_labels & right_labels)
                proper_surface_evidence = (
                    len(surface_tokens) >= 2 and compatible_roles and not generic
                )
                neighborhood_evidence = len(neighborhood_overlap) >= 2
                accepted = (
                    not numeric_surface
                    and not generic
                    and (proper_surface_evidence or neighborhood_evidence)
                )
                if accepted:
                    union(left, right)
                decisions.append(
                    {
                        "surface": surface,
                        "node_type": node_type,
                        "left_uid": left,
                        "right_uid": right,
                        "left_document": native.nodes[left].get("document_id"),
                        "right_document": native.nodes[right].get("document_id"),
                        "left_roles": sorted(left_labels),
                        "right_roles": sorted(right_labels),
                        "state_jaccard": state_jaccard,
                        "neighborhood_overlap": neighborhood_overlap,
                        "accepted": accepted,
                    }
                )

    roots: dict[str, int] = {}
    mapping: dict[str, str] = {}
    for node in sorted(occurrences):
        root = find(node)
        if root not in roots:
            roots[root] = len(roots)
        surface = canonical_entity_name(str(native.nodes[node].get("name", node)))
        mapping[node] = f"{surface}::c{roots[root]}"
    return mapping, decisions


def aggregate_projection(unreconciled, mapping: Mapping[str, str]):
    import networkx as nx

    result = nx.Graph()
    for node, data in unreconciled.nodes(data=True):
        target = mapping[node]
        if target not in result:
            result.add_node(
                target,
                name=data.get("name", node),
                node_type=data.get("node_type"),
                occurrences=0,
                documents=set(),
            )
        result.nodes[target]["occurrences"] += 1
        result.nodes[target]["documents"].add(data.get("document_id"))
    for left, right, data in unreconciled.edges(data=True):
        new_left, new_right = mapping[left], mapping[right]
        if new_left == new_right:
            continue
        weight = int(data.get("weight", 1))
        if result.has_edge(new_left, new_right):
            result[new_left][new_right]["weight"] += weight
        else:
            result.add_edge(new_left, new_right, weight=weight)
    return result


def network_statistics(name: str, graph) -> dict[str, Any]:
    import networkx as nx
    import numpy as np

    simple = nx.Graph(graph.to_undirected()) if graph.is_directed() else nx.Graph(graph)
    sizes = sorted((len(group) for group in nx.connected_components(simple)))
    n = simple.number_of_nodes()
    return {
        "graph": name,
        "nodes": n,
        "edges": simple.number_of_edges(),
        "components": len(sizes),
        "giant": sizes[-1] if sizes else 0,
        "giant_fraction": sizes[-1] / n if sizes and n else 0.0,
        "isolates": nx.number_of_isolates(simple),
        "component_min": sizes[0] if sizes else 0,
        "component_median": float(np.median(sizes)) if sizes else 0.0,
        "component_mean": float(np.mean(sizes)) if sizes else 0.0,
        "component_max": sizes[-1] if sizes else 0,
        "average_clustering": nx.average_clustering(simple) if n else 0.0,
        "degree_assortativity": (
            nx.degree_assortativity_coefficient(simple)
            if simple.number_of_edges() > 1
            else float("nan")
        ),
    }


def top_centralities(graph, top_n: int = 10, seed: int = 232) -> dict[str, list]:
    import networkx as nx

    simple = nx.Graph(graph)
    degree = dict(simple.degree(weight="weight"))
    pagerank = nx.pagerank(simple, weight="weight") if simple else {}
    sample = min(500, simple.number_of_nodes())
    between = (
        nx.betweenness_centrality(simple, k=sample, seed=seed, weight=None)
        if sample and simple.number_of_edges()
        else {}
    )

    def highest(values: Mapping[str, float]) -> list[tuple[str, float]]:
        return sorted(values.items(), key=lambda item: (-item[1], item[0]))[:top_n]

    return {
        "weighted_degree": highest(degree),
        "pagerank": highest(pagerank),
        "betweenness": highest(between),
    }


def validate_plan(plan: object) -> dict[str, Any]:
    errors: list[str] = []
    if not isinstance(plan, list) or not plan:
        return {"valid": False, "errors": ["plan must be a nonempty list"], "edges": []}
    edges: list[tuple[int, int]] = []
    for step_number, row in enumerate(plan, start=1):
        if not isinstance(row, Mapping):
            errors.append(f"Q{step_number} is not an object")
            continue
        question = str(row.get("question", "")).strip()
        if not question:
            errors.append(f"Q{step_number} has no question")
        for raw_parent in PLACEHOLDER.findall(question):
            parent = int(raw_parent)
            edges.append((parent, step_number))
            if parent >= step_number or parent < 1:
                errors.append(f"Q{step_number} has invalid reference Q{parent}")
    return {"valid": not errors, "errors": errors, "edges": sorted(set(edges))}


def decomposition_metrics(
    predicted: Mapping[str, Sequence[Mapping[str, Any]]],
    reviewed: Mapping[str, Sequence[Mapping[str, Any]]],
) -> dict[str, float]:
    valid, count_exact, true_flags, predicted_flags = [], [], [], []
    edge_tp = edge_fp = edge_fn = 0
    evaluated = 0
    for qid, gold in reviewed.items():
        if qid not in predicted:
            continue
        evaluated += 1
        plan = predicted[qid]
        valid.append(float(validate_plan(plan)["valid"]))
        count_exact.append(float(len(plan) == len(gold)))
        gold_edges = set(map(tuple, validate_plan(gold)["edges"]))
        plan_edges = set(map(tuple, validate_plan(plan)["edges"]))
        edge_tp += len(gold_edges & plan_edges)
        edge_fp += len(plan_edges - gold_edges)
        edge_fn += len(gold_edges - plan_edges)
        for left, right in zip(plan, gold):
            predicted_flags.append(bool(left.get("requires_retrieval", True)))
            true_flags.append(bool(right.get("requires_retrieval", True)))
    precision = edge_tp / (edge_tp + edge_fp) if edge_tp + edge_fp else 0.0
    recall = edge_tp / (edge_tp + edge_fn) if edge_tp + edge_fn else 0.0
    return {
        "questions": evaluated,
        "valid_plan_rate": sum(valid) / evaluated if evaluated else 0.0,
        "subquestion_count_exact": sum(count_exact) / evaluated if evaluated else 0.0,
        "dependency_edge_precision": precision,
        "dependency_edge_recall": recall,
        "dependency_edge_f1": (
            2 * precision * recall / (precision + recall)
            if precision + recall
            else 0.0
        ),
        "retrieval_reasoning_accuracy": (
            sum(a == b for a, b in zip(predicted_flags, true_flags))
            / len(true_flags)
            if true_flags
            else 0.0
        ),
    }


def retrieval_branches(plan: Sequence[Mapping[str, Any]]) -> tuple[list[list[int]], list[str]]:
    """Extract single-parent retrieval chains and report unsupported joins."""

    retrieval = {
        index: row
        for index, row in enumerate(plan, start=1)
        if bool(row.get("requires_retrieval", True))
    }
    parent: dict[int, int | None] = {}
    warnings: list[str] = []
    for index, row in retrieval.items():
        refs = [int(value) for value in PLACEHOLDER.findall(str(row["question"]))]
        retrieval_refs = [value for value in refs if value in retrieval]
        if len(retrieval_refs) > 1:
            warnings.append(
                f"Q{index} is a multi-parent retrieval join; evidence is taken from its parent branches"
            )
            continue
        if refs and not retrieval_refs:
            warnings.append(
                f"Q{index} depends on a reasoning result and is not issued as retrieval"
            )
            continue
        parent[index] = retrieval_refs[0] if retrieval_refs else None
    children: dict[int, set[int]] = {index: set() for index in parent}
    for child, parent_id in parent.items():
        if parent_id in children:
            children[parent_id].add(child)
    leaves = [index for index in parent if not children[index]]
    branches: list[list[int]] = []
    for leaf in sorted(leaves):
        path, current, seen = [], leaf, set()
        while current is not None and current not in seen:
            seen.add(current)
            path.append(current)
            current = parent.get(current)
        path.reverse()
        if path and path not in branches:
            branches.append(path)
    return branches, warnings


def _chain_key(chain: Mapping[str, Any]) -> tuple:
    return (
        -float(chain["score"]),
        tuple(step["qa_uid"] for step in chain["steps"]),
    )


def prune_chains(
    chains: Sequence[Mapping[str, Any]],
    beam_width: int,
    *,
    unique_answers: bool = True,
) -> list[dict[str, Any]]:
    kept: list[dict[str, Any]] = []
    seen: set[str] = set()
    for chain in sorted(chains, key=_chain_key):
        answer_key = normalize_text(chain["steps"][-1]["answer"])
        if unique_answers and answer_key in seen:
            continue
        seen.add(answer_key)
        kept.append(dict(chain))
        if len(kept) == beam_width:
            break
    return kept


def pending_branch_queries(
    plan: Sequence[Mapping[str, Any]],
    state: Mapping[str, Any],
) -> list[str]:
    """Instantiate the next query once for every current parent beam."""

    step_ids = state["step_ids"]
    hop_index = int(state["hop_index"])
    if hop_index >= len(step_ids):
        return []
    if hop_index > 0 and not state["beams"]:
        return []
    step_id = step_ids[hop_index]
    template = str(plan[step_id - 1]["question"])
    parents = state["beams"] or [{"steps": [], "answers": {}, "score": 1.0}]
    queries: list[str] = []
    for parent in parents:
        answers = parent["answers"]

        def replace_placeholder(match: re.Match[str]) -> str:
            reference = int(match.group(1))
            if reference not in answers:
                raise KeyError(f"unresolved Q{reference} in {template}")
            return str(answers[reference])

        queries.append(PLACEHOLDER.sub(replace_placeholder, template))
    return queries


def advance_branch_one_hop(
    plan: Sequence[Mapping[str, Any]],
    state: dict[str, Any],
    retrieve,
    *,
    beam_width: int,
    candidates_per_hop: int,
    unique_answers: bool = True,
    score_rule: str = "geometric_mean",
) -> None:
    queries = pending_branch_queries(plan, state)
    if not queries:
        return
    step_id = state["step_ids"][state["hop_index"]]
    template = str(plan[step_id - 1]["question"])
    parents = state["beams"] or [{"steps": [], "answers": {}, "score": 1.0}]
    expansions: list[dict[str, Any]] = []
    for parent, concrete in zip(parents, queries):
        for candidate in retrieve(concrete, candidates_per_hop):
            steps = [*parent["steps"], candidate]
            scores = [max(float(item["score"]), 1e-12) for item in steps]
            score = (
                scores[-1]
                if score_rule == "last_hop"
                else math.exp(sum(math.log(value) for value in scores) / len(scores))
            )
            expansions.append(
                {
                    "steps": steps,
                    "answers": {**parent["answers"], step_id: candidate["answer"]},
                    "score": score,
                }
            )
    state["beams"] = prune_chains(
        expansions, beam_width, unique_answers=unique_answers
    )
    state["trace"].append(
        {
            "hop": state["hop_index"] + 1,
            "step_id": step_id,
            "template": template,
            "issued_queries": queries,
            "expansions": len(expansions),
            "kept": [
                {
                    "qa_ids": [step["qa_uid"] for step in beam["steps"]],
                    "answers": [step["answer"] for step in beam["steps"]],
                    "score": beam["score"],
                }
                for beam in state["beams"]
            ],
        }
    )
    state["hop_index"] += 1


def result_from_branch_states(
    branches: Sequence[dict[str, Any]], warnings: Sequence[str]
) -> dict[str, Any]:
    evidence: dict[str, dict[str, Any]] = {}
    for branch in branches:
        for beam in branch["beams"]:
            for step in beam["steps"]:
                previous = evidence.get(step["qa_uid"])
                if previous is None or step["score"] > previous["score"]:
                    evidence[step["qa_uid"]] = step
    return {
        "branches": [branch["step_ids"] for branch in branches],
        "warnings": list(warnings),
        "chains": [beam for branch in branches for beam in branch["beams"]],
        "branch_traces": [
            {"step_ids": branch["step_ids"], "hops": branch["trace"]}
            for branch in branches
        ],
        "evidence": list(evidence.values()),
    }


def run_branch(
    plan: Sequence[Mapping[str, Any]],
    step_ids: Sequence[int],
    retrieve,
    *,
    beam_width: int,
    candidates_per_hop: int,
    unique_answers: bool = True,
    score_rule: str = "geometric_mean",
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    beams: list[dict[str, Any]] = []
    trace: list[dict[str, Any]] = []
    for hop, step_id in enumerate(step_ids, start=1):
        template = str(plan[step_id - 1]["question"])
        parents = beams if beams else [{"steps": [], "answers": {}, "score": 1.0}]
        expansions: list[dict[str, Any]] = []
        issued_queries: list[str] = []
        for parent in parents:
            answers = parent["answers"]

            def replace(match: re.Match[str]) -> str:
                referenced = int(match.group(1))
                if referenced not in answers:
                    raise KeyError(f"unresolved Q{referenced} in {template}")
                return str(answers[referenced])

            concrete = PLACEHOLDER.sub(replace, template)
            issued_queries.append(concrete)
            for candidate in retrieve(concrete, candidates_per_hop):
                steps = [*parent["steps"], candidate]
                scores = [max(float(item["score"]), 1e-12) for item in steps]
                if score_rule == "last_hop":
                    score = scores[-1]
                else:
                    score = math.exp(sum(math.log(value) for value in scores) / len(scores))
                expansions.append(
                    {
                        "steps": steps,
                        "answers": {**answers, step_id: candidate["answer"]},
                        "score": score,
                    }
                )
        beams = prune_chains(
            expansions, beam_width, unique_answers=unique_answers
        )
        trace.append(
            {
                "hop": hop,
                "step_id": step_id,
                "template": template,
                "issued_queries": issued_queries,
                "expansions": len(expansions),
                "kept": [
                    {
                        "qa_ids": [step["qa_uid"] for step in beam["steps"]],
                        "answers": [step["answer"] for step in beam["steps"]],
                        "score": beam["score"],
                    }
                    for beam in beams
                ],
            }
        )
        if not beams:
            break
    return beams, trace


def _discarded_linear_execute_plan(
    plan: Sequence[Mapping[str, Any]],
    retrieve,
    config: RunConfig,
    *,
    unique_answers: bool = True,
    score_rule: str = "geometric_mean",
) -> dict[str, Any]:
    """Pre-audit linear approximation retained only to document the old failure.

    The production and answer-key paths call :func:`execute_panini_plan`.
    Keeping this private helper makes old cached traces readable without
    presenting the approximation as PANINI.
    """
    branches, warnings = retrieval_branches(plan)
    all_beams, branch_traces = [], []
    for branch in branches:
        beams, trace = run_branch(
            plan,
            branch,
            retrieve,
            beam_width=config.beam_width,
            candidates_per_hop=config.candidates_per_hop,
            unique_answers=unique_answers,
            score_rule=score_rule,
        )
        all_beams.append(beams)
        branch_traces.append({"step_ids": branch, "hops": trace})

    # The final answerer receives the best chain from each independent branch.
    evidence: dict[str, dict[str, Any]] = {}
    for beams in all_beams:
        if not beams:
            continue
        for step in beams[0]["steps"]:
            previous = evidence.get(step["qa_uid"])
            if previous is None or step["score"] > previous["score"]:
                evidence[step["qa_uid"]] = step
    return {
        "branches": branches,
        "warnings": warnings,
        "chains": [beam for beams in all_beams for beam in beams],
        "branch_traces": branch_traces,
        "evidence": list(evidence.values()),
    }


def retrieval_components(
    plan: Sequence[Mapping[str, Any]],
) -> list[list[int]]:
    """Return retrieval-node connected components in topological order.

    This is the student-scale equivalent of
    ``ChainFollowingMultiHopQA.identify_reasoning_chains``.  Unlike the old
    linear-branch helper, a component may contain a converging DAG node with
    two or more retrieval parents.
    """

    retrieval_ids = {
        index
        for index, row in enumerate(plan, start=1)
        if bool(row.get("requires_retrieval", True))
    }
    parents: dict[int, list[int]] = {}
    children: dict[int, list[int]] = {index: [] for index in retrieval_ids}
    for index in sorted(retrieval_ids):
        refs = [int(value) for value in PLACEHOLDER.findall(str(plan[index - 1]["question"]))]
        parents[index] = [value for value in refs if value in retrieval_ids]
        for parent in parents[index]:
            children[parent].append(index)

    components: list[list[int]] = []
    visited: set[int] = set()
    for start in sorted(retrieval_ids):
        if start in visited:
            continue
        component: set[int] = set()
        stack = [start]
        while stack:
            current = stack.pop()
            if current in component:
                continue
            component.add(current)
            stack.extend(parents.get(current, []))
            stack.extend(children.get(current, []))
        visited.update(component)
        indegree = {
            node: len([parent for parent in parents[node] if parent in component])
            for node in component
        }
        queue = sorted(node for node, degree in indegree.items() if degree == 0)
        ordered: list[int] = []
        while queue:
            node = queue.pop(0)
            ordered.append(node)
            for child in sorted(children.get(node, [])):
                if child not in component:
                    continue
                indegree[child] -= 1
                if indegree[child] == 0:
                    queue.append(child)
                    queue.sort()
        if len(ordered) != len(component):
            raise ValueError(f"retrieval dependency cycle in {sorted(component)}")
        # The research implementation treats singleton retrieval questions as
        # the simple-retrieval fallback rather than as a reasoning chain.
        if len(ordered) > 1:
            components.append(ordered)
    return components


def _panini_harmonic_mean(scores: Sequence[float]) -> float:
    valid = [float(score) for score in scores if float(score) > 1e-6]
    return len(valid) / sum(1.0 / score for score in valid) if valid else 1e-6


def _panini_chain_score(
    steps: Sequence[Mapping[str, Any]], score_rule: str = "geometric_mean"
) -> float:
    normalized = [
        max(1e-6, min(1.0, 0.5 * (float(step.get("score", 0.0)) + 1.0)))
        for step in steps
    ]
    if not normalized:
        return 1e-6
    if score_rule == "last_hop":
        return normalized[-1]
    return math.exp(sum(math.log(score) for score in normalized) / len(normalized))


def _panini_state_key(state: Mapping[str, Any]) -> tuple[Any, ...]:
    return (
        -float(state.get("chain_score", 0.0)),
        -float(state.get("last_hop_score", 0.0)),
        tuple(step["qa_uid"] for step in state.get("steps", [])),
    )


def _combine_parent_states(
    parent_groups: Sequence[Sequence[Mapping[str, Any]]],
    beam_width: int,
    threshold: float,
) -> list[dict[str, Any]]:
    combinations: list[tuple[float, tuple[Mapping[str, Any], ...]]] = []
    for states in product(*parent_groups):
        score = _panini_harmonic_mean(
            [float(state.get("chain_score", 0.5)) for state in states]
        )
        combinations.append((score, states))
    combinations.sort(
        key=lambda item: (
            -item[0],
            tuple(
                tuple(step["qa_uid"] for step in state.get("steps", []))
                for state in item[1]
            ),
        )
    )
    selected = [item for item in combinations[:beam_width] if item[0] >= threshold]
    if not selected and combinations:
        selected = combinations[:1]
    merged: list[dict[str, Any]] = []
    for harmonic_score, states in selected:
        answers: dict[int, Any] = {}
        steps: list[dict[str, Any]] = []
        for state in states:
            answers.update(state.get("answers", {}))
            for step in state.get("steps", []):
                # Match the research code: parent evidence lists are merged
                # before chain scoring. Formatting deduplicates final evidence.
                steps.append(dict(step))
        merged.append(
            {
                "answers": answers,
                "steps": steps,
                "chain_score": harmonic_score,
                "pre_combination_score": harmonic_score,
            }
        )
    return merged


def execute_panini_plan(
    plan: Sequence[Mapping[str, Any]],
    retrieve_qa,
    config: RunConfig,
    *,
    original_question: str | None = None,
    unique_answers: bool = True,
    score_rule: str = "geometric_mean",
) -> dict[str, Any]:
    """Execute the real PANINI DAG semantics with course-scale retrieval.

    ``retrieve_qa`` returns QA records, each retaining all answer names, stable
    answer IDs, role states, and one reranker score.  Intermediate hops keep
    the best state per answer entity.  A converging retrieval node combines
    parent beams by harmonic mean.  The final retrieval hop keeps QA-level
    alternatives and evidence is deduplicated from *all* final beams.
    """

    components = retrieval_components(plan)
    if not components:
        fallback_query = original_question or next(
            (
                str(row.get("question", ""))
                for row in plan
                if bool(row.get("requires_retrieval", True))
            ),
            "",
        )
        candidates = (
            list(retrieve_qa(fallback_query, config.candidates_per_hop))
            if fallback_query
            else []
        )
        evidence = [dict(candidate) for candidate in candidates]
        chains = [
            {
                "answers": {1: list(candidate.get("answer_names", []))},
                "steps": [dict(candidate)],
                "chain_score": _panini_chain_score([candidate], score_rule),
                "last_hop_score": float(candidate.get("score", 0.0)),
            }
            for candidate in candidates[: config.beam_width]
        ]
        return {
            "components": [],
            "warnings": [],
            "chains": chains,
            "component_traces": [],
            "fallback": True,
            "fallback_query": fallback_query,
            "evidence": evidence,
        }
    final_states: list[dict[str, Any]] = []
    component_traces: list[dict[str, Any]] = []
    warnings: list[str] = []
    for component in components:
        completed: dict[int, list[dict[str, Any]]] = {}
        step_traces: list[dict[str, Any]] = []
        for position, step_id in enumerate(component):
            template = str(plan[step_id - 1]["question"])
            refs = [int(value) for value in PLACEHOLDER.findall(template)]
            dependencies = [value for value in refs if value in component]
            unresolved_reasoning = [value for value in refs if value not in component]
            if unresolved_reasoning:
                warnings.append(
                    f"Q{step_id} references non-retrieval steps {unresolved_reasoning}"
                )
            if not dependencies:
                prior_states = [
                    {"answers": {}, "steps": [], "chain_score": 1.0}
                ]
            elif len(dependencies) == 1:
                prior_states = [dict(state) for state in completed.get(dependencies[0], [])]
            else:
                parent_groups = [completed.get(parent, [])[: config.beam_width] for parent in dependencies]
                prior_states = _combine_parent_states(
                    parent_groups,
                    config.beam_width,
                    config.multi_dependency_threshold,
                ) if all(parent_groups) else []

            expansions: list[dict[str, Any]] = []
            issued_queries: list[str] = []
            final_hop = position == len(component) - 1
            for state in prior_states:
                missing = [ref for ref in refs if ref not in state["answers"]]
                if missing:
                    warnings.append(
                        f"Q{step_id} could not resolve placeholders {missing}; state skipped"
                    )
                    continue
                concrete = PLACEHOLDER.sub(
                    lambda match: (
                        ", ".join(state["answers"][int(match.group(1))])
                        if isinstance(state["answers"][int(match.group(1))], list)
                        else str(state["answers"][int(match.group(1))])
                    ),
                    template,
                )
                issued_queries.append(concrete)
                candidates = list(retrieve_qa(concrete, config.candidates_per_hop))
                if final_hop:
                    for candidate in candidates:
                        answers = dict(state["answers"])
                        names = list(candidate.get("answer_names", []))
                        if names:
                            # The research executor keeps the full final answer
                            # list because no later placeholder consumes it.
                            answers[step_id] = names
                        steps = [*state["steps"], dict(candidate)]
                        expansions.append(
                            {
                                "answers": answers,
                                "steps": steps,
                                "chain_score": _panini_chain_score(steps, score_rule),
                                "last_hop_score": float(candidate.get("score", 0.0)),
                            }
                        )
                elif unique_answers:
                    per_entity: dict[str, dict[str, Any]] = {}
                    for candidate in candidates:
                        names = list(candidate.get("answer_names", []))
                        ids = list(candidate.get("answer_ids", []))
                        for answer_index, answer in enumerate(names):
                            if not answer:
                                continue
                            entity_key = (
                                str(ids[answer_index])
                                if answer_index < len(ids) and ids[answer_index]
                                else normalize_text(answer)
                            )
                            answers = dict(state["answers"])
                            answers[step_id] = answer
                            steps = [*state["steps"], dict(candidate)]
                            new_state = {
                                "answers": answers,
                                "steps": steps,
                                "chain_score": _panini_chain_score(steps, score_rule),
                                "last_hop_score": float(candidate.get("score", 0.0)),
                            }
                            previous = per_entity.get(entity_key)
                            if previous is None or _panini_state_key(new_state) < _panini_state_key(previous):
                                per_entity[entity_key] = new_state
                    expansions.extend(per_entity.values())
                else:
                    for candidate in candidates:
                        names = list(candidate.get("answer_names", []))
                        for answer in names:
                            if not answer:
                                continue
                            answers = dict(state["answers"])
                            answers[step_id] = answer
                            steps = [*state["steps"], dict(candidate)]
                            expansions.append(
                                {
                                    "answers": answers,
                                    "steps": steps,
                                    "chain_score": _panini_chain_score(steps, score_rule),
                                    "last_hop_score": float(candidate.get("score", 0.0)),
                                }
                            )
            expansions.sort(key=_panini_state_key)
            beams = expansions[: config.beam_width]
            completed[step_id] = beams
            step_traces.append(
                {
                    "step_id": step_id,
                    "template": template,
                    "dependencies": dependencies,
                    "parent_states": len(prior_states),
                    "issued_queries": issued_queries,
                    "expansions": len(expansions),
                    "final_hop": final_hop,
                    "kept": [
                        {
                            "qa_ids": [step["qa_uid"] for step in state["steps"]],
                            "answers": dict(state["answers"]),
                            "score": state["chain_score"],
                        }
                        for state in beams
                    ],
                }
            )
        component_final = completed.get(component[-1], []) if component else []
        final_states.extend(component_final)
        component_traces.append({"step_ids": component, "steps": step_traces})

    evidence: dict[str, dict[str, Any]] = {}
    for state in final_states:
        for step in state.get("steps", []):
            previous = evidence.get(step["qa_uid"])
            if previous is None or float(step.get("score", 0.0)) > float(previous.get("score", 0.0)):
                evidence[step["qa_uid"]] = dict(step)
    return {
        "components": components,
        "warnings": warnings,
        "chains": final_states,
        "component_traces": component_traces,
        "evidence": list(evidence.values()),
    }


def evidence_answers(question: Mapping[str, Any]) -> list[str]:
    answers: list[str] = []
    for evidence in question.get("evidences", []):
        if isinstance(evidence, Mapping):
            answers.append(str(evidence.get("answer", "")))
        elif isinstance(evidence, (list, tuple)) and evidence:
            answers.append(str(evidence[-1]))
    return [answer for answer in answers if answer]


def gold_qa_ids(question: Mapping[str, Any], qa_rows: Sequence[Mapping[str, Any]]) -> set[str]:
    selected: set[str] = set()
    default_documents = set(
        question.get("supporting_document_ids", question.get("context_document_ids", []))
    )
    for task in atomic_tasks(question):
        documents = {task["document_id"]} if task["document_id"] else default_documents
        answer = normalize_text(task["answer"])
        candidates = [
            row
            for row in qa_rows
            if row.get("document_id") in documents
            and answer in {
                normalize_text(value) for value in row.get("answer_names", [])
            }
        ]
        if not candidates and documents != set(question.get("context_document_ids", [])):
            candidates = [
                row
                for row in qa_rows
                if row.get("document_id") in question.get("context_document_ids", [])
                and answer in {
                    normalize_text(value) for value in row.get("answer_names", [])
                }
            ]
        query_tokens = set(WORD.findall(task["question"].casefold())) - STOPWORDS

        def relevance(row: Mapping[str, Any]) -> tuple[float, str]:
            text = f"{row.get('question', '')} {row.get('verb_phrase', '')}"
            row_tokens = set(WORD.findall(text.casefold())) - STOPWORDS
            overlap = len(query_tokens & row_tokens) / max(len(query_tokens | row_tokens), 1)
            return overlap, str(row["qa_uid"])

        if candidates:
            best = sorted(candidates, key=lambda row: (-relevance(row)[0], relevance(row)[1]))[0]
            selected.add(str(best["qa_uid"]))
    return selected


def atomic_tasks(question: Mapping[str, Any]) -> list[dict[str, str]]:
    tasks: list[dict[str, str]] = []
    previous: dict[int, str] = {}
    for index, evidence in enumerate(question.get("evidences", []), start=1):
        if isinstance(evidence, Mapping):
            query = str(evidence.get("question", ""))
            query = re.sub(
                r"#(\d+)",
                lambda match: previous.get(int(match.group(1)), match.group(0)),
                query,
            )
            answer = str(evidence.get("answer", ""))
            document_id = str(evidence.get("document_id", ""))
        else:
            subject, relation, answer = map(str, evidence)
            query = f"{subject} >> {relation}"
            document_id = ""
        previous[index] = answer
        tasks.append(
            {"question": query, "answer": answer, "document_id": document_id}
        )
    return tasks


def stratified_sample(
    questions: Sequence[Mapping[str, Any]],
    group_field: str,
    size: int = 20,
    seed: int = 232,
) -> list[Mapping[str, Any]]:
    groups: dict[str, list[Mapping[str, Any]]] = defaultdict(list)
    for row in questions:
        groups[str(row[group_field])].append(row)
    rng = random.Random(seed)
    selected: list[Mapping[str, Any]] = []
    ordered_groups = sorted(groups)
    base, remainder = divmod(size, len(ordered_groups))
    for index, group in enumerate(ordered_groups):
        rows = sorted(groups[group], key=lambda row: str(row["question_id"]))
        rng.shuffle(rows)
        selected.extend(rows[: base + (index < remainder)])
    return sorted(selected, key=lambda row: str(row["question_id"]))


def evaluate_ranked_ids(ranked_ids: Sequence[str], relevant: set[str], k: int) -> dict[str, float]:
    hits = [index for index, qa_uid in enumerate(ranked_ids[:k], start=1) if qa_uid in relevant]
    return {
        "recall": len(set(ranked_ids[:k]) & relevant) / len(relevant) if relevant else 0.0,
        "reciprocal_rank": 1.0 / hits[0] if hits else 0.0,
    }


def run_decomposition_stage(
    jobs: Sequence[tuple[str, Any, Sequence[Mapping[str, Any]], Path]],
    config: RunConfig,
    *,
    model_name: str = "yigitturali/GSW-QA-Decomposer-Qwen3-4B",
) -> None:
    """Run all datasets with one 4-bit decomposer and append after each question."""

    import torch
    from panini_course.qwen_models import QwenDecomposer

    if not torch.cuda.is_available():
        raise RuntimeError("The full decomposition stage requires a GPU runtime")
    dtype = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"
    prompt_path = jobs[0][1].root / "models/decomposition_prompt.txt"
    model = QwenDecomposer(
        model_name,
        prompt_path,
        quantized=True,
        dtype=dtype,
        device_map="auto",
    )
    for dataset, _package, questions, cache_root in jobs:
        path = cache_root / "decompositions.jsonl"
        completed = {row["question_id"] for row in read_jsonl(path)}
        for question in questions:
            qid = str(question["question_id"])
            if qid in completed:
                continue
            started = time.perf_counter()
            raw = ""
            try:
                raw = model.generate_raw(
                    str(question["question"]),
                    max_new_tokens=config.max_new_tokens,
                )
                plan = model.parse_response(raw)
                validation = validate_plan(plan)
                error = None
            except Exception as exception:  # retained in the trace by design
                plan, validation, error = [], {"valid": False, "errors": []}, repr(exception)
            append_jsonl(
                path,
                {
                    "dataset": dataset,
                    "question_id": qid,
                    "question": question["question"],
                    "raw_response": raw,
                    "predicted_decomposition": plan,
                    "decomposition_valid": validation["valid"],
                    "decomposition_errors": validation["errors"],
                    "error": error,
                    "seconds": time.perf_counter() - started,
                },
            )
            print(
                f"[decompose] {dataset} {len(read_jsonl(path))}/100 {qid}",
                flush=True,
            )
    del model
    release_gpu()


class NeuralRetrievalCache:
    """Persistent exact-query cache for all Q5--Q7 ranking variants."""

    def __init__(self, package, cache_root: Path, config: RunConfig, encoder, reranker):
        import numpy as np
        from panini_course import (
            BM25Index,
            DenseIndex,
            DualRetriever,
            QueryEmbeddingStore,
            TfidfIndex,
        )

        self.np = np
        self.package = package
        self.cache_root = Path(cache_root)
        self.config = config
        self.encoder = encoder
        self.reranker = reranker
        self.qa_rows = package.qa_pairs()
        self.qa_by_id = {row["qa_uid"]: row for row in self.qa_rows}
        self.entity_rows = package.entities()
        root = package.root
        self.entity_bm25 = BM25Index.load(
            root / "indices/entity_bm25.joblib",
            root / "indices/entity_ids.json",
            source="entity_bm25",
        )
        self.qa_bm25 = BM25Index.load(
            root / "indices/qa_bm25.joblib",
            root / "indices/qa_ids.json",
            source="qa_bm25",
        )
        self.qa_tfidf = TfidfIndex.load(
            root / "indices/qa_tfidf.npz",
            root / "indices/qa_tfidf_vectorizer.joblib",
            root / "indices/qa_ids.json",
            source="qa_tfidf",
        )
        self.qa_dense = DenseIndex.load(
            root / "indices/qa_qwen3_8b_ip.faiss",
            root / "indices/qa_ids.json",
            source="qa_dense",
        )
        self.query_store = QueryEmbeddingStore.load(
            root / "embeddings/query_embeddings.npy",
            root / "embeddings/query_ids.json",
            root / "embeddings/queries.jsonl",
        )
        self.dual = DualRetriever(
            entity_index=self.entity_bm25,
            qa_index=self.qa_dense,
            entity_rows=self.entity_rows,
            qa_rows=self.qa_rows,
        )
        self.query_path = self.cache_root / "retrieval_cache.jsonl"
        self.records = {
            row["query_key"]: row for row in read_jsonl(self.query_path)
        }
        self.vector_manifest_path = self.cache_root / "query_vector_manifest.jsonl"
        self.vector_records = {
            row["query_key"]: row for row in read_jsonl(self.vector_manifest_path)
        }
        self.vector_memory: dict[str, Any] = {}

    def vector(self, query: str):
        key = normalize_text(query)
        if key in self.vector_memory:
            return self.vector_memory[key]
        try:
            vector = self.query_store.get(query)
        except KeyError:
            if key in self.vector_records:
                vector = self.np.load(
                    self.cache_root / self.vector_records[key]["file"]
                )
            else:
                if self.encoder is None:
                    raise RuntimeError(
                        f"Query vector is not prepared for: {query}"
                    )
                started = time.perf_counter()
                vector = self.encoder.encode([query], max_length=256)[0]
                relative = f"query_vectors/query_{len(self.vector_records):06d}.npy"
                target = self.cache_root / relative
                target.parent.mkdir(parents=True, exist_ok=True)
                self.np.save(target, vector)
                record = {
                    "query_key": key,
                    "query": query,
                    "file": relative,
                    "seconds": time.perf_counter() - started,
                }
                append_jsonl(self.vector_manifest_path, record)
                self.vector_records[key] = record
        self.vector_memory[key] = vector
        return vector

    def _expand_answers(self, hits, score_name: str = "score") -> list[dict[str, Any]]:
        candidates: list[dict[str, Any]] = []
        for hit in hits:
            row = self.qa_by_id[hit.item_id]
            for answer in row.get("answer_names", []):
                candidates.append(
                    {
                        "qa_uid": row["qa_uid"],
                        "answer": answer,
                        "question": row["question"],
                        "document_id": row["document_id"],
                        "score": float(getattr(hit, score_name, hit.score)),
                        "rank": int(hit.rank),
                        "source": hit.source,
                    }
                )
        return candidates

    def compute(self, query: str) -> dict[str, Any]:
        from panini_course import reciprocal_rank_fusion

        key = normalize_text(query)
        if key in self.records:
            return self.records[key]
        if self.reranker is None:
            raise RuntimeError(f"Reranker result is not prepared for: {query}")
        started = time.perf_counter()
        pool = self.config.retrieval_pool
        bm25 = self.qa_bm25.search(query, pool)
        tfidf = self.qa_tfidf.search(query, pool)
        dense = self.qa_dense.search(self.vector(query), pool)
        rrf = reciprocal_rank_fusion(
            [tfidf, bm25, dense],
            rank_constant=self.config.rrf_constant,
            top_k=pool,
        )
        dual = self.dual.search(
            query,
            query_vector=self.vector(query),
            entity_top_k=self.config.entity_top_k,
            qa_top_k=self.config.qa_top_k,
            fused_top_k=pool,
            rank_constant=self.config.rrf_constant,
        )
        rows = [self.qa_by_id[hit.item_id] for hit in dual]
        rerank_scores = self.reranker.score(
            query,
            [row["search_text"] for row in rows],
            batch_size=self.config.rerank_batch_size,
        )
        reranked: dict[str, list[dict[str, Any]]] = {
            "reranker_only": [],
            "retrieval_only": [],
            "dual_hybrid": [],
        }
        for hit, row, reranker_score in zip(dual, rows, rerank_scores):
            retrieval_score = 1.0 / hit.rank
            hybrid = (
                self.config.retrieval_weight * retrieval_score
                + (1.0 - self.config.retrieval_weight) * float(reranker_score)
            )
            base = {
                "qa_uid": row["qa_uid"],
                "question": row["question"],
                "document_id": row["document_id"],
                "retrieval_rank": hit.rank,
                "reranker_score": float(reranker_score),
                "routes": list(hit.metadata.get("fusion_sources", [])),
                "source": "dual",
            }
            for answer in row.get("answer_names", []):
                reranked["reranker_only"].append(
                    {**base, "answer": answer, "score": float(reranker_score)}
                )
                reranked["retrieval_only"].append(
                    {**base, "answer": answer, "score": retrieval_score}
                )
                reranked["dual_hybrid"].append(
                    {**base, "answer": answer, "score": hybrid}
                )
        for name in reranked:
            reranked[name].sort(
                key=lambda row: (-row["score"], row["qa_uid"], row["answer"])
            )
        rankings = {
            "bm25": self._expand_answers(bm25),
            "dense": self._expand_answers(dense),
            "rrf": self._expand_answers(rrf),
            **reranked,
        }
        record = {
            "query_key": key,
            "query": query,
            "rankings": rankings,
            "seconds": time.perf_counter() - started,
        }
        append_jsonl(self.query_path, record)
        self.records[key] = record
        return record

    def candidates(self, query: str, top_k: int, backend: str = "dual_hybrid"):
        return self.compute(query)["rankings"][backend][:top_k]

    def qa_candidates(
        self, query: str, top_k: int, backend: str = "reranker_only"
    ) -> list[dict[str, Any]]:
        """Return top unique QA pairs without flattening their answer entities.

        RICR expands answer entities only at intermediate hops. Flattening QA
        rows before the executor changes both top-k and final-hop behavior, so
        this adapter reconstructs the record shape used by the research code.
        """

        key = normalize_text(query)
        if backend in {"bm25", "dense", "rrf"} and key not in self.records:
            # Baseline ablations do not require a cross-encoder. Avoid scoring
            # an unused dual pool merely to obtain a BM25/dense/RRF ranking.
            from panini_course import reciprocal_rank_fusion

            pool = self.config.retrieval_pool
            bm25 = self.qa_bm25.search(query, pool)
            if backend == "bm25":
                hits = bm25
            else:
                dense = self.qa_dense.search(self.vector(query), pool)
                if backend == "dense":
                    hits = dense
                else:
                    tfidf = self.qa_tfidf.search(query, pool)
                    hits = reciprocal_rank_fusion(
                        [tfidf, bm25, dense],
                        rank_constant=self.config.rrf_constant,
                        top_k=pool,
                    )
            ranking = self._expand_answers(hits)
        else:
            ranking = self.compute(query)["rankings"][backend]
        candidates: list[dict[str, Any]] = []
        seen: set[str] = set()
        for hit in ranking:
            qa_uid = str(hit["qa_uid"])
            if qa_uid in seen:
                continue
            seen.add(qa_uid)
            row = self.qa_by_id[qa_uid]
            candidates.append(
                {
                    "qa_uid": qa_uid,
                    "question": row["question"],
                    "document_id": row["document_id"],
                    "answer_names": list(row.get("answer_names", [])),
                    # GSW node IDs (e1, e2, ...) are document-local. Namespacing
                    # prevents unrelated e1 nodes from collapsing during the
                    # per-entity beam step. Cross-document identity remains an
                    # explicit reconciliation limitation for students to study.
                    "answer_ids": [
                        f"{row['document_id']}::{local_id}"
                        for local_id in row.get("answer_local_ids", [])
                    ],
                    "answer_role_states": list(
                        row.get("answer_role_states", [])
                    ),
                    "score": float(hit["score"]),
                    "retrieval_rank": hit.get("retrieval_rank", hit.get("rank")),
                    "routes": list(hit.get("routes", [])),
                    "source": hit.get("source", backend),
                }
            )
            if len(candidates) >= top_k:
                break
        return candidates


def run_neural_retrieval_stage(
    jobs: Sequence[tuple[str, Any, Sequence[Mapping[str, Any]], Path]],
    config: RunConfig,
    *,
    backend: str = "dual_hybrid",
    warm_gold_atomic_queries: bool = True,
    run_ablations: bool = True,
) -> None:
    """Run Qwen query encoding, reranking, and RICR with one model pair."""

    import torch
    from panini_course.qwen_models import QwenQueryEncoder, QwenReranker

    if not torch.cuda.is_available():
        raise RuntimeError("The full retrieval stage requires a GPU runtime")
    if torch.cuda.get_device_properties(0).total_memory < 14 * 2**30:
        raise RuntimeError("The 4-bit encoder/reranker stage needs a 15–16 GiB GPU")
    dtype = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"
    encoder = QwenQueryEncoder(quantized=True, dtype=dtype, device_map="auto")
    reranker = QwenReranker(
        model_name=config.reranker_model,
        quantized=True,
        dtype=dtype,
        device_map="auto",
        max_length=config.rerank_max_length,
    )
    for dataset, package, questions, cache_root in jobs:
        plans = {
            row["question_id"]: row for row in read_jsonl(cache_root / "decompositions.jsonl")
        }
        trace_path = cache_root / "traces.jsonl"
        completed = {row["question_id"] for row in read_jsonl(trace_path)}
        retrieval = NeuralRetrievalCache(package, cache_root, config, encoder, reranker)
        if warm_gold_atomic_queries:
            for public_question in package.questions("public"):
                for task in atomic_tasks(public_question):
                    retrieval.compute(task["question"])
        for question in questions:
            qid = str(question["question_id"])
            if qid in completed:
                continue
            plan_record = plans.get(qid)
            if not plan_record or not plan_record.get("decomposition_valid"):
                append_jsonl(
                    trace_path,
                    {
                        "dataset": dataset,
                        "question_id": qid,
                        "question": question["question"],
                        "error": "missing or invalid decomposition",
                        "chains": [],
                        "evidence": [],
                        "seconds": 0.0,
                    },
                )
                continue
            started = time.perf_counter()
            try:
                result = execute_panini_plan(
                    plan_record["predicted_decomposition"],
                    lambda query, k: retrieval.qa_candidates(query, k, backend),
                    config,
                    original_question=str(question["question"]),
                )
                error = None
            except Exception as exception:
                result = {
                    "components": [],
                    "warnings": [],
                    "chains": [],
                    "component_traces": [],
                    "evidence": [],
                }
                error = repr(exception)
            append_jsonl(
                trace_path,
                {
                    "dataset": dataset,
                    "question_id": qid,
                    "question": question["question"],
                    "predicted_decomposition": plan_record["predicted_decomposition"],
                    "backend": backend,
                    "reranker_model": config.reranker_model,
                    **result,
                    "error": error,
                    "seconds": time.perf_counter() - started,
                },
            )
            print(
                f"[retrieve] {dataset} {len(read_jsonl(trace_path))}/100 {qid}",
                flush=True,
            )
        if run_ablations:
            group_field = "type" if dataset == "2wiki" else "hop_count"
            subset = stratified_sample(
                package.questions("public"), group_field, size=20, seed=config.seed
            )
            ablation_path = cache_root / "ablation_traces.jsonl"
            completed_ablations = {
                (row["configuration"], row["question_id"])
                for row in read_jsonl(ablation_path)
            }
            ablations = [
                ("default", config, "dual_hybrid", True, "geometric_mean"),
                ("beam_1", replace(config, beam_width=1), "dual_hybrid", True, "geometric_mean"),
                ("beam_3", replace(config, beam_width=3), "dual_hybrid", True, "geometric_mean"),
                ("k_5", replace(config, candidates_per_hop=5), "dual_hybrid", True, "geometric_mean"),
                ("unique_off", config, "dual_hybrid", False, "geometric_mean"),
                ("last_hop", config, "dual_hybrid", True, "last_hop"),
                ("parent_threshold_off", replace(config, multi_dependency_threshold=0.0), "dual_hybrid", True, "geometric_mean"),
                ("bm25", config, "bm25", True, "geometric_mean"),
                ("dense", config, "dense", True, "geometric_mean"),
                ("rrf", config, "rrf", True, "geometric_mean"),
            ]
            qa_rows = package.qa_pairs()
            for name, ablation_config, ablation_backend, unique, score_rule in ablations:
                for question in subset:
                    qid = str(question["question_id"])
                    if (name, qid) in completed_ablations:
                        continue
                    plan_record = plans.get(qid)
                    if not plan_record or not plan_record.get("decomposition_valid"):
                        continue
                    started = time.perf_counter()
                    result = execute_panini_plan(
                        plan_record["predicted_decomposition"],
                        lambda query, k, selected=ablation_backend: retrieval.qa_candidates(
                            query, k, selected
                        ),
                        ablation_config,
                        original_question=str(question["question"]),
                        unique_answers=unique,
                        score_rule=score_rule,
                    )
                    append_jsonl(
                        ablation_path,
                        {
                            "dataset": dataset,
                            "configuration": name,
                            "question_id": qid,
                            "question": question["question"],
                            "backend": ablation_backend,
                            "reranker_model": config.reranker_model,
                            "beam_width": ablation_config.beam_width,
                            "candidates_per_hop": ablation_config.candidates_per_hop,
                            "unique_answers": unique,
                            "score_rule": score_rule,
                            **result,
                            **score_trace(question, result, qa_rows),
                            "seconds": time.perf_counter() - started,
                        },
                    )
                print(f"[ablation] {dataset} {name} complete", flush=True)
    del encoder, reranker
    release_gpu()


def _scheduled_retrieval_configuration(
    jobs: Sequence[tuple[str, Any, Sequence[Mapping[str, Any]], Path]],
    config: RunConfig,
    *,
    output_name: str,
    configuration_name: str,
    backend: str,
    unique_answers: bool,
    score_rule: str,
    warm_gold_atomic_queries: bool = False,
) -> None:
    """Execute the faithful DAG engine while loading one GPU model at a time.

    Each round replays the deterministic executor from cached rankings.  A
    callback records the first uncached queries exposed at the current DAG
    frontier; those queries are embedded and reranked in two separate GPU
    phases.  Replaying avoids maintaining a second, subtly different RICR
    implementation solely for low-memory Colab runtimes.
    """

    import torch
    from panini_course.qwen_models import QwenQueryEncoder, QwenReranker

    if not torch.cuda.is_available():
        raise RuntimeError("The full retrieval stage requires a GPU runtime")
    dtype = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"
    total_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
    reranker_model = select_reranker_model(config, total_gib)
    if reranker_model == config.free_colab_reranker_model:
        config = replace(
            config,
            rerank_batch_size=config.free_colab_rerank_batch_size,
            rerank_max_length=config.free_colab_rerank_max_length,
        )
    print(f"[low-memory] {total_gib:.1f} GiB GPU; reranker={reranker_model}", flush=True)

    contexts: list[dict[str, Any]] = []
    for dataset, package, questions, cache_root in jobs:
        output_path = cache_root / output_name
        completed = {
            str(row["question_id"])
            for row in read_jsonl(output_path)
            if row.get("configuration", configuration_name) == configuration_name
        }
        plans = {
            str(row["question_id"]): row
            for row in read_jsonl(cache_root / "decompositions.jsonl")
        }
        pending: dict[str, dict[str, Any]] = {}
        for question in questions:
            qid = str(question["question_id"])
            if qid in completed:
                continue
            plan_record = plans.get(qid)
            if not plan_record or not plan_record.get("decomposition_valid"):
                append_jsonl(
                    output_path,
                    {
                        "dataset": dataset,
                        "configuration": configuration_name,
                        "question_id": qid,
                        "question": question["question"],
                        "error": "missing or invalid decomposition",
                        "chains": [],
                        "evidence": [],
                        "seconds": 0.0,
                    },
                )
                continue
            pending[qid] = {
                "question": question,
                "plan": plan_record["predicted_decomposition"],
                "started": time.perf_counter(),
            }
        contexts.append(
            {
                "dataset": dataset,
                "package": package,
                "cache_root": cache_root,
                "output_path": output_path,
                "pending": pending,
                "qa_rows": package.qa_pairs(),
            }
        )

    round_number = 0
    while any(context["pending"] for context in contexts):
        round_number += 1
        discoveries: dict[str, list[str]] = {}
        finished: list[tuple[dict[str, Any], str, dict[str, Any]]] = []
        for context in contexts:
            cache = NeuralRetrievalCache(
                context["package"], context["cache_root"], config, None, None
            )
            missing: list[str] = []
            if warm_gold_atomic_queries and round_number == 1:
                for public_question in context["package"].questions("public"):
                    for task in atomic_tasks(public_question):
                        if normalize_text(task["question"]) not in cache.records:
                            missing.append(task["question"])
            for qid, state in context["pending"].items():
                state_missing: list[str] = []

                def cached_retrieve(query, k, selected=backend):
                    if selected in {"bm25", "dense", "rrf"}:
                        return cache.qa_candidates(query, k, selected)
                    if normalize_text(query) not in cache.records:
                        state_missing.append(query)
                        return []
                    return cache.qa_candidates(query, k, selected)

                result = execute_panini_plan(
                    state["plan"],
                    cached_retrieve,
                    config,
                    original_question=str(state["question"]["question"]),
                    unique_answers=unique_answers,
                    score_rule=score_rule,
                )
                missing.extend(state_missing)
                if not state_missing:
                    finished.append((context, qid, result))
            discoveries[context["dataset"]] = list(dict.fromkeys(missing))

        for context, qid, result in finished:
            state = context["pending"].pop(qid)
            cache = NeuralRetrievalCache(
                context["package"], context["cache_root"], config, None, None
            )
            issued_queries = {
                query
                for component in result.get("component_traces", [])
                for step in component.get("steps", [])
                for query in step.get("issued_queries", [])
            }
            if result.get("fallback_query"):
                issued_queries.add(result["fallback_query"])
            query_seconds = sum(
                float(cache.records.get(normalize_text(query), {}).get("seconds", 0.0))
                + float(cache.vector_records.get(normalize_text(query), {}).get("seconds", 0.0))
                for query in issued_queries
            )
            record = {
                "dataset": context["dataset"],
                "configuration": configuration_name,
                "question_id": qid,
                "question": state["question"]["question"],
                "predicted_decomposition": state["plan"],
                "backend": backend,
                "reranker_model": reranker_model,
                "beam_width": config.beam_width,
                "candidates_per_hop": config.candidates_per_hop,
                "unique_answers": unique_answers,
                "score_rule": score_rule,
                **result,
                "error": None,
                "seconds": query_seconds or (time.perf_counter() - state["started"]),
                "scheduled_wall_seconds": time.perf_counter() - state["started"],
            }
            if output_name == "ablation_traces.jsonl":
                record.update(score_trace(state["question"], result, context["qa_rows"]))
            append_jsonl(context["output_path"], record)

        new_queries = sum(map(len, discoveries.values()))
        print(f"[low-memory round {round_number}] ranking misses: {new_queries}", flush=True)
        if not new_queries:
            # All remaining questions must have been written above. This guard
            # prevents an accidental infinite loop if that invariant changes.
            if any(context["pending"] for context in contexts):
                raise RuntimeError("RICR scheduler stalled without discovering a query")
            break

        vector_misses: dict[str, list[str]] = {}
        for context in contexts:
            cache = NeuralRetrievalCache(
                context["package"], context["cache_root"], config, None, None
            )
            missing_vectors: list[str] = []
            for query in discoveries[context["dataset"]]:
                try:
                    cache.vector(query)
                except RuntimeError:
                    missing_vectors.append(query)
            vector_misses[context["dataset"]] = missing_vectors
        if sum(map(len, vector_misses.values())):
            encoder = QwenQueryEncoder(quantized=True, dtype=dtype, device_map="auto")
            for context in contexts:
                cache = NeuralRetrievalCache(
                    context["package"], context["cache_root"], config, encoder, None
                )
                for query in vector_misses[context["dataset"]]:
                    cache.vector(query)
            del cache, encoder
            release_gpu()

        try:
            reranker = QwenReranker(
                model_name=reranker_model,
                quantized=True,
                dtype=dtype,
                device_map="auto",
                max_length=config.rerank_max_length,
            )
        except RuntimeError as exception:
            if (
                "out of memory" not in str(exception).casefold()
                or reranker_model == config.free_colab_reranker_model
            ):
                raise
            release_gpu()
            reranker_model = config.free_colab_reranker_model
            config = replace(
                config,
                rerank_batch_size=config.free_colab_rerank_batch_size,
                rerank_max_length=config.free_colab_rerank_max_length,
            )
            print(
                f"[low-memory] 8B OOM; falling back to {reranker_model}",
                flush=True,
            )
            reranker = QwenReranker(
                model_name=reranker_model,
                quantized=True,
                dtype=dtype,
                device_map="auto",
                max_length=config.rerank_max_length,
            )
        for context in contexts:
            cache = NeuralRetrievalCache(
                context["package"], context["cache_root"], config, None, reranker
            )
            for query in discoveries[context["dataset"]]:
                if normalize_text(query) not in cache.records:
                    cache.compute(query)
        del cache, reranker
        release_gpu()


def run_neural_retrieval_stage_low_memory(
    jobs: Sequence[tuple[str, Any, Sequence[Mapping[str, Any]], Path]],
    config: RunConfig,
    *,
    run_ablations: bool = True,
) -> None:
    """Free-tier path: replay DAG frontiers from supplied vectors and cached rankings."""

    _scheduled_retrieval_configuration(
        jobs,
        config,
        output_name="traces.jsonl",
        configuration_name="default",
        backend="dual_hybrid",
        unique_answers=True,
        score_rule="geometric_mean",
        warm_gold_atomic_queries=True,
    )
    if not run_ablations:
        return
    ablations = [
        ("default", config, "dual_hybrid", True, "geometric_mean"),
        ("beam_1", replace(config, beam_width=1), "dual_hybrid", True, "geometric_mean"),
        ("beam_3", replace(config, beam_width=3), "dual_hybrid", True, "geometric_mean"),
        ("k_5", replace(config, candidates_per_hop=5), "dual_hybrid", True, "geometric_mean"),
        ("unique_off", config, "dual_hybrid", False, "geometric_mean"),
        ("last_hop", config, "dual_hybrid", True, "last_hop"),
        ("parent_threshold_off", replace(config, multi_dependency_threshold=0.0), "dual_hybrid", True, "geometric_mean"),
        ("bm25", config, "bm25", True, "geometric_mean"),
        ("dense", config, "dense", True, "geometric_mean"),
        ("rrf", config, "rrf", True, "geometric_mean"),
    ]
    for name, selected_config, backend, unique, score_rule in ablations:
        subset_jobs = []
        for dataset, package, _questions, cache_root in jobs:
            group_field = "type" if dataset == "2wiki" else "hop_count"
            subset = stratified_sample(
                package.questions("public"),
                group_field,
                size=20,
                seed=config.seed,
            )
            subset_jobs.append((dataset, package, subset, cache_root))
        _scheduled_retrieval_configuration(
            subset_jobs,
            selected_config,
            output_name="ablation_traces.jsonl",
            configuration_name=name,
            backend=backend,
            unique_answers=unique,
            score_rule=score_rule,
        )


def score_trace(question: Mapping[str, Any], trace: Mapping[str, Any], qa_rows) -> dict[str, Any]:
    from panini_course.metrics import exact_match

    selected_ids = {row["qa_uid"] for row in trace.get("evidence", [])}
    all_chain_steps = [
        step for chain in trace.get("chains", []) for step in chain.get("steps", [])
    ]
    all_chain_ids = {step["qa_uid"] for step in all_chain_steps}
    gold_answer_sequence = [normalize_text(answer) for answer in evidence_answers(question)]
    retained_answers = {
        normalize_text(answer)
        for chain in trace.get("chains", [])
        for answer_value in chain.get("answers", {}).values()
        for answer in (answer_value if isinstance(answer_value, list) else [answer_value])
    }
    complete_chain = all(answer in retained_answers for answer in gold_answer_sequence)
    gold_ids = gold_qa_ids(question, qa_rows)
    document_by_qa = {str(row["qa_uid"]): row.get("document_id") for row in qa_rows}
    support_docs = set(question.get("supporting_document_ids", [])) or {
        document_by_qa[qa_uid]
        for qa_uid in gold_ids
        if document_by_qa.get(qa_uid)
    }
    selected_docs = {row.get("document_id") for row in trace.get("evidence", [])}
    gold_answers = [question["answer"], *question.get("answer_aliases", [])]
    return {
        "supporting_qa_recall": len(all_chain_ids & gold_ids) / len(gold_ids) if gold_ids else 0.0,
        "supporting_document_recall": (
            len(selected_docs & support_docs) / len(support_docs) if support_docs else 0.0
        ),
        "complete_chain_recovery": float(complete_chain),
        "answer_in_selected_evidence": max(
            (
                exact_match(answer, gold_answers)
                for row in trace.get("evidence", [])
                for answer in row.get("answer_names", [row.get("answer", "")])
            ),
            default=0.0,
        ),
        "selected_evidence_count": len(selected_ids),
        "surviving_chains": len(trace.get("chains", [])),
        "unique_current_answers": len(
            {
                normalize_text(answer)
                for chain in trace.get("chains", [])
                if chain.get("steps")
                for answer in chain["steps"][-1].get(
                    "answer_names", [chain["steps"][-1].get("answer", "")]
                )
            }
        ),
    }


def format_answer_evidence(row: Mapping[str, Any]) -> str:
    """Format one QA record exactly as the PANINI answer prompt expects."""

    answers = ", ".join(row.get("answer_names", [])) or str(row.get("answer", ""))
    roles = ", ".join(row.get("answer_role_states", []))
    formatted = f"Q: {row['question']} A: {answers}"
    return f"{formatted} {roles}" if roles else formatted


def run_answer_stage(
    jobs: Sequence[tuple[str, Any, Sequence[Mapping[str, Any]], Path]],
    output_root: Path,
    config: RunConfig,
    *,
    run_ablations: bool = True,
) -> None:
    import torch
    from panini_course.metrics import exact_match, token_f1
    from panini_course.qwen_models import QwenAnswerer

    if not torch.cuda.is_available():
        raise RuntimeError("The full answer stage requires a GPU runtime")
    dtype = "bfloat16" if torch.cuda.get_device_capability(0)[0] >= 8 else "float16"
    answerer = QwenAnswerer(quantized=True, dtype=dtype, device_map="auto")
    for dataset, package, questions, cache_root in jobs:
        plans = {
            row["question_id"]: row for row in read_jsonl(cache_root / "decompositions.jsonl")
        }
        traces = {row["question_id"]: row for row in read_jsonl(cache_root / "traces.jsonl")}
        prediction_path = cache_root / "answers.jsonl"
        completed = {row["question_id"] for row in read_jsonl(prediction_path)}
        qa_rows = package.qa_pairs()
        public_ids = {row["question_id"] for row in package.questions("public")}
        for question in questions:
            qid = str(question["question_id"])
            if qid in completed:
                continue
            trace = traces.get(qid, {"chains": [], "evidence": [], "seconds": 0.0})
            evidence = [
                format_answer_evidence(row) for row in trace.get("evidence", [])
            ]
            started = time.perf_counter()
            answer_trace = answerer.answer_with_trace(
                str(question["question"]), evidence
            )
            prediction = str(answer_trace["answer"])
            answer_seconds = time.perf_counter() - started
            context = "\n".join(evidence)
            context_tokens = len(answerer.tokenizer(context).input_ids)
            plan_record = plans.get(qid, {})
            issued_queries = {
                query
                for component in trace.get("component_traces", [])
                for step in component.get("steps", [])
                for query in step.get("issued_queries", [])
            }
            if trace.get("fallback_query"):
                issued_queries.add(trace["fallback_query"])
            record: dict[str, Any] = {
                "dataset": dataset,
                "split": "development" if qid in public_ids else "held_out",
                "question_id": qid,
                "question": question["question"],
                "predicted_decomposition": plan_record.get("predicted_decomposition", []),
                "decomposition_valid": bool(plan_record.get("decomposition_valid", False)),
                "retrieval_backend": trace.get("backend", "dual_hybrid"),
                "reranker_model": trace.get(
                    "reranker_model", config.reranker_model
                ),
                "beam_width": config.beam_width,
                "candidates_per_hop": config.candidates_per_hop,
                "chains": [
                    {
                        "qa_ids": [step["qa_uid"] for step in chain.get("steps", [])],
                        "answers": [
                            list(step.get("answer_names", [step.get("answer", "")]))
                            for step in chain.get("steps", [])
                        ],
                        "hop_scores": [step["score"] for step in chain.get("steps", [])],
                        "chain_score": chain.get("chain_score", chain.get("score", 0.0)),
                    }
                    for chain in trace.get("chains", [])
                ],
                "evidence_qa_ids": [row["qa_uid"] for row in trace.get("evidence", [])],
                "answer_evidence": evidence,
                "predicted_answer": prediction,
                "answer_response": answer_trace["response"],
                "latency_ms": {
                    "decomposition": 1000 * float(plan_record.get("seconds", 0.0)),
                    "retrieval_ricr": 1000 * float(trace.get("seconds", 0.0)),
                    "answer": 1000 * answer_seconds,
                },
                "answer_context_tokens": context_tokens,
                "evidence_count": len(evidence),
                "reranked_candidate_count": len(issued_queries) * config.retrieval_pool,
            }
            if qid in public_ids:
                aliases = [question["answer"], *question.get("answer_aliases", [])]
                record.update(score_trace(question, trace, qa_rows))
                record.update(
                    {
                        "gold_answer": question["answer"],
                        "exact_match": exact_match(prediction, aliases),
                        "token_f1": token_f1(prediction, aliases),
                    }
                )
            append_jsonl(prediction_path, record)
            print(
                f"[answer] {dataset} {len(read_jsonl(prediction_path))}/100 {qid}",
                flush=True,
            )

        predictions = read_jsonl(prediction_path)
        for row in predictions:
            row.setdefault("reranker_model", config.reranker_model)
        write_jsonl(prediction_path, predictions)
        write_jsonl(
            output_root / "results" / f"{dataset}_dev.jsonl",
            (row for row in predictions if row["split"] == "development"),
        )
        write_jsonl(
            output_root / "predictions" / f"{dataset}_heldout.jsonl",
            (row for row in predictions if row["split"] == "held_out"),
        )
        if run_ablations:
            ablation_rows = read_jsonl(cache_root / "ablation_traces.jsonl")
            ablation_answer_path = cache_root / "ablation_answers.jsonl"
            completed_ablations = {
                (row["configuration"], row["question_id"])
                for row in read_jsonl(ablation_answer_path)
            }
            question_by_id = {
                row["question_id"]: row for row in package.questions("public")
            }
            for row in ablation_rows:
                key = (row["configuration"], row["question_id"])
                if key in completed_ablations:
                    continue
                evidence = [
                    format_answer_evidence(item) for item in row.get("evidence", [])
                ]
                started = time.perf_counter()
                prediction = answerer.answer(row["question"], evidence)
                question = question_by_id[row["question_id"]]
                aliases = [question["answer"], *question.get("answer_aliases", [])]
                refreshed_metrics = score_trace(question, row, qa_rows)
                append_jsonl(
                    ablation_answer_path,
                    {
                        "dataset": dataset,
                        "configuration": row["configuration"],
                        "question_id": row["question_id"],
                        "reranker_model": row.get(
                            "reranker_model", config.reranker_model
                        ),
                        "predicted_answer": prediction,
                        "exact_match": exact_match(prediction, aliases),
                        "token_f1": token_f1(prediction, aliases),
                        "answer_seconds": time.perf_counter() - started,
                        "supporting_qa_recall": refreshed_metrics["supporting_qa_recall"],
                        "complete_chain_recovery": refreshed_metrics["complete_chain_recovery"],
                        "retrieval_seconds": row["seconds"],
                        "evidence_count": len(evidence),
                    },
                )
            ablation_predictions = read_jsonl(ablation_answer_path)
            for row in ablation_predictions:
                row.setdefault("reranker_model", config.reranker_model)
            write_jsonl(ablation_answer_path, ablation_predictions)
    del answerer
    release_gpu()


def result_summary(records: Sequence[Mapping[str, Any]], group_field: str) -> list[dict[str, Any]]:
    import numpy as np

    groups: dict[str, list[Mapping[str, Any]]] = defaultdict(list)
    for row in records:
        groups[str(row.get(group_field, "all"))].append(row)
    output = []
    for group, rows in sorted(groups.items()):
        totals = [sum(item["latency_ms"].values()) for item in rows]
        output.append(
            {
                group_field: group,
                "questions": len(rows),
                "decomposition_valid": float(np.mean([row["decomposition_valid"] for row in rows])),
                "supporting_qa_recall": float(np.mean([row.get("supporting_qa_recall", 0) for row in rows])),
                "supporting_document_recall": float(
                    np.mean([row.get("supporting_document_recall", 0) for row in rows])
                ),
                "complete_chain_recovery": float(np.mean([row.get("complete_chain_recovery", 0) for row in rows])),
                "surviving_chains_mean": float(
                    np.mean([row.get("surviving_chains", 0) for row in rows])
                ),
                "unique_current_answers_mean": float(
                    np.mean([row.get("unique_current_answers", 0) for row in rows])
                ),
                "EM": float(np.mean([row.get("exact_match", 0) for row in rows])),
                "F1": float(np.mean([row.get("token_f1", 0) for row in rows])),
                "latency_mean_ms": float(np.mean(totals)),
                "latency_p95_ms": float(np.percentile(totals, 95)),
                "evidence_mean": float(np.mean([row.get("evidence_count", 0) for row in rows])),
                "reranked_candidates_mean": float(
                    np.mean([row.get("reranked_candidate_count", 0) for row in rows])
                ),
                "answer_tokens_mean": float(np.mean([row.get("answer_context_tokens", 0) for row in rows])),
            }
        )
    return output


def write_environment(path: str | Path, config: RunConfig) -> None:
    import importlib.metadata

    packages = {}
    for name in ("torch", "transformers", "bitsandbytes", "faiss-cpu", "networkx"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = "not installed"
    payload = {
        "python": sys.version,
        "platform": platform.platform(),
        "gpu": gpu_snapshot(),
        "packages": packages,
        "models": {
            "decomposer": "yigitturali/GSW-QA-Decomposer-Qwen3-4B",
            "query_encoder": "Qwen/Qwen3-Embedding-8B",
            "reranker": "Qwen/Qwen3-Reranker-8B",
            "answerer": "Qwen/Qwen3-4B",
        },
        "configuration": asdict(config),
    }
    Path(path).write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display
from panini_course import CoursePackage

CONFIG = RunConfig()
packages = {name: CoursePackage(PACKAGE_ROOTS[name]) for name in DATASETS}
jobs = []
for name, package in packages.items():
    questions = (package.questions('public') + package.questions('held_out'))[QUESTION_SLICE]
    cache_root = CACHE_BASE / name
    cache_root.mkdir(parents=True, exist_ok=True)
    jobs.append((name, package, questions, cache_root))
print(gpu_snapshot(), {'questions_in_this_run': sum(len(job[2]) for job in jobs)})

## Question 1 — understand and verify the two packages (6 points)

The trusted boundary is an identifier audit, not a row-count audit.
`entity_uid` and `qa_uid` carry document, GSW, and local-node
provenance. Two arrays can have equal lengths while one is
permuted, duplicated, or missing an ID; row counts would pass while
retrieval silently attaches a vector to the wrong QA. The checks
below therefore require uniqueness *and* set equality between
metadata IDs and embedding IDs. Held-out rows are also inspected by
field name, because a nonempty input file is correct but any answer
or evidence field would violate the evaluation boundary. The final
table also distinguishes named-node coverage from grounded-QA
coverage. Three development questions have atomic answers present
as entity nodes but not as answers of a grounded QA edge; they are
retained and counted as operational retrieval failures rather than
silently removed after observing labels.

In [ ]:
audit_rows, schema_examples = [], {}
for dataset, package in packages.items():
    rows, examples = audit_package(package, dataset)
    audit_rows.extend(rows)
    schema_examples[dataset] = examples
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
for dataset, examples in schema_examples.items():
    print(f'\n{dataset} question example:', json.dumps(examples['question'], indent=2)[:1600])
    print('entity example:', json.dumps(examples['entity'], indent=2)[:1200])
    print('QA example:', json.dumps(examples['qa'], indent=2)[:1200])

qa_coverage_exceptions = []
for dataset, package in packages.items():
    qa_rows = package.qa_pairs()
    for question in package.questions('public'):
        if not gold_qa_ids(question, qa_rows):
            qa_coverage_exceptions.append({
                'dataset': dataset,
                'question_id': question['question_id'],
                'question': question['question'],
                'named_node_but_no_grounded_QA_answer': True,
            })
display(pd.DataFrame(qa_coverage_exceptions))

**Explanation.** A question ID identifies the evaluation
unit; a document ID locates provenance; an entity UID identifies one
document-local occurrence; and a QA UID identifies one grounded
verb-question-answer edge. None is interchangeable with a Python
list position. A plausible undetected failure is sorting
`qa_pairs.jsonl` without applying the same permutation to
`qa_embeddings.npy`: shape checks still pass, FAISS still returns
valid row numbers, and every displayed answer is nevertheless
attached to the wrong vector. The set-equality and ID-to-row maps
above prevent that failure.

## Questions 2–3 — construct, reconcile, and analyze the network (24 points)

The native multigraph records what each GSW actually asserted. The
unreconciled projection keeps document-local occurrence IDs. The
exact-surface projection is an intentionally aggressive sensitivity
condition. The conservative mapping accepts the same normalized
surface and node type only when it also has identity-bearing role or
neighborhood evidence; nationality, profession, date, number,
genre, and other attribute values are explicitly blocked. These
projections are analysis objects and are never passed to retrieval.

In [ ]:
from panini_course.graph import (
    build_entity_projection, build_native_gsw_graph,
    build_unreconciled_entity_projection,
)

graph_sets, network_rows, decision_sets = {}, [], {}
for dataset, package in packages.items():
    native = build_native_gsw_graph(package.gsw_paths())
    unreconciled = build_unreconciled_entity_projection(native)
    exact = build_entity_projection(native)
    mapping, decisions = conservative_entity_mapping(native)
    conservative = aggregate_projection(unreconciled, mapping)
    graph_sets[dataset] = {
        'native': native, 'unreconciled': unreconciled,
        'exact_surface': exact, 'conservative': conservative,
    }
    decision_sets[dataset] = decisions
    network_rows.extend(
        {'dataset': dataset, **network_statistics(name, graph)}
        for name, graph in graph_sets[dataset].items()
    )
    # Edge direction, provenance, and cross-document non-merge checks.
    assert all(data['edge_type'] == 'qa' for *_, data in native.edges(data=True))
    assert all('document_id' in data for *_, data in native.edges(data=True))
    assert all('::' in node for node in unreconciled.nodes)
network_table = pd.DataFrame(network_rows)
display(network_table)

In [ ]:
# Fixed, independently labeled 30-pair reconciliation audit.
from io import StringIO

manual_audit = pd.read_csv(StringIO('dataset,surface,left_uid,right_uid,reference_label,rationale\n2wiki,American,doc_1012::gsw_1012_0.json::e4,doc_1136::gsw_1136_0.json::e4,incorrect,"A nationality value is not a shared person identity and becomes a false hub."\n2wiki,Italian,doc_1129::gsw_1129_0.json::e3,doc_2586::gsw_2586_0.json::e4,incorrect,"The occurrences are nationality attributes attached to different people."\n2wiki,film director,doc_1140::gsw_1140_0.json::e6,doc_1665::gsw_1665_0.json::e6,incorrect,"This is a profession value rather than one named director."\n2wiki,actor,doc_1466::gsw_1466_0.json::e4,doc_1979::gsw_1979_0.json::e8,incorrect,"This is a profession value rather than one named actor."\n2wiki,One More Time,doc_1238::gsw_1238_0.json::e1,doc_1239::gsw_1239_0.json::e1,incorrect,"The first occurrence is an ambiguous title and the second is a music group."\n2wiki,20th Century Fox,doc_1214::gsw_1214_0.json::e13,doc_1219::gsw_1219_0.json::e12,correct,"Both occurrences describe the same film-studio organization."\n2wiki,Academy Awards,doc_2147::gsw_2147_0.json::e26,doc_5394::gsw_5394_0.json::e12,correct,"Both occurrences identify the same film-award institution."\n2wiki,Adelaide Herrmann,doc_4691::gsw_4691_0.json::e1,doc_4692::gsw_4692_0.json::e6,correct,"Both are the historical magician Adelaide Herrmann."\n2wiki,Alexander Herrmann,doc_4691::gsw_4691_0.json::e5,doc_4692::gsw_4692_0.json::e1,correct,"Both are the historical magician Alexander Herrmann."\n2wiki,New York City,doc_1219::gsw_1219_0.json::e9,doc_4855::gsw_4855_0.json::e10,correct,"Premiere and residence roles refer to the same city."\n2wiki,United States,doc_1221::gsw_1221_0.json::e10,doc_15::gsw_15_0.json::e4,correct,"Country and release-location mentions refer to the same country."\n2wiki,comedy film,doc_12::gsw_12_0.json::e5,doc_1981::gsw_1981_0.json::e4,incorrect,"This is a genre value shared by unrelated films."\n2wiki,Paris,doc_1665::gsw_1665_0.json::e23,doc_2194::gsw_2194_0.json::e3,correct,"Acting-location and birthplace roles refer to Paris, France."\n2wiki,Barcelona,doc_1236::gsw_1236_0.json::e12,doc_1402::gsw_1402_0.json::e10,correct,"Both location roles identify the city rather than the football club."\n2wiki,Japan,doc_10::gsw_10_0.json::e2,doc_1481::gsw_1481_0.json::e1,correct,"Both occurrences identify the country."\nmusique,American,doc_10295::gsw_10295_0.json::e2,doc_10307::gsw_10307_0.json::e4,incorrect,"A nationality value should not connect otherwise unrelated people."\nmusique,Charles Mingus,doc_1042::gsw_1042_0.json::e3,doc_2908::gsw_2908_0.json::e2,correct,"Both occurrences describe the jazz musician and composer."\nmusique,Ben Affleck,doc_10048::gsw_10048_0.json::e7,doc_10051::gsw_10051_0.json::e10,correct,"Both person records identify the same actor."\nmusique,Florida Panthers,doc_2747::gsw_2747_0.json::e3,doc_2749::gsw_2749_0.json::e4,correct,"Both sports-team records identify the NHL team."\nmusique,National Football League,doc_10307::gsw_10307_0.json::e6,doc_2756::gsw_2756_0.json::e6,correct,"Organization and sports-league roles identify the NFL."\nmusique,New York City,doc_1035::gsw_1035_0.json::e3,doc_1490::gsw_1490_0.json::e2,correct,"City and licensed-location roles refer to the same city."\nmusique,United States,doc_10045::gsw_10045_0.json::e3,doc_1036::gsw_1036_0.json::e2,correct,"Both country records identify the United States."\nmusique,19th century,doc_3644::gsw_3644_0.json::e2,doc_6022::gsw_6022_0.json::e3,incorrect,"The shared time-period value is not an entity instance."\nmusique,African American,doc_2662::gsw_2662_0.json::e5,doc_962::gsw_962_0.json::e7,incorrect,"The occurrences are ethnicity attributes attached to different people."\nmusique,Atlantic Ocean,doc_2042::gsw_2042_0.json::e3,doc_2045::gsw_2045_0.json::e4,correct,"Both geographical records identify the same ocean."\nmusique,Chicago,doc_124::gsw_124_0.json::e10,doc_1284::gsw_1284_0.json::e2,correct,"Both location records identify the city."\nmusique,London,doc_10077::gsw_10077_0.json::e17,doc_1093::gsw_1093_0.json::e4,correct,"Filming-location and capital-city roles identify London, England."\nmusique,Paris,doc_10083::gsw_10083_0.json::e5,doc_165::gsw_165_0.json::e5,correct,"Death-place and appearance-location roles identify Paris, France."\nmusique,Australia,doc_1247::gsw_1247_0.json::e4,doc_1264::gsw_1264_0.json::e5,uncertain,"One record emphasizes the continent and the other the nation; the intended granularity is unclear."\nmusique,Soviet Union,doc_10880::gsw_10880_0.json::e6,doc_10889::gsw_10889_0.json::e5,correct,"Both records identify the same historical political entity."\n'))
decision_lookup = {
    (dataset, row['left_uid'], row['right_uid']): row['accepted']
    for dataset, rows in decision_sets.items() for row in rows
}
manual_audit['conservative_decision'] = [
    decision_lookup.get((row.dataset, row.left_uid, row.right_uid), False)
    for row in manual_audit.itertuples()
]
certain = manual_audit[manual_audit.reference_label != 'uncertain']
precision = np.mean([
    label == 'correct' for label in certain[certain.conservative_decision].reference_label
]) if certain.conservative_decision.any() else np.nan
recall = np.mean(
    certain[certain.reference_label == 'correct'].conservative_decision
)
display(manual_audit)
print({'audited_pairs': len(manual_audit),
       'uncertain': int((manual_audit.reference_label == 'uncertain').sum()),
       'estimated_precision_excluding_uncertain': precision,
       'reference_recall_on_audited_correct_pairs': recall})

missed_aliases = pd.DataFrame([
    {'dataset': 'musique', 'left': 'USA', 'right': 'United States',
     'reason': 'surface normalization does not expand abbreviations'},
    {'dataset': 'musique', 'left': 'UK', 'right': 'United Kingdom',
     'reason': 'surface normalization does not expand abbreviations'},
])
display(missed_aliases)

In [ ]:
# Degree PMF and CCDF: two panels per dataset, four plots total.
fig, axes = plt.subplots(len(graph_sets), 2, figsize=(13, 5 * len(graph_sets)))
for row_index, (dataset, variants) in enumerate(graph_sets.items()):
    for name, graph in variants.items():
        simple = nx.Graph(graph)
        degrees = np.array([degree for _, degree in simple.degree()])
        positive = degrees[degrees > 0]
        values, counts = np.unique(positive, return_counts=True)
        axes[row_index, 0].loglog(values, counts / counts.sum(), marker='.', label=name)
        ordered = np.sort(positive)
        ccdf = 1 - np.arange(len(ordered)) / len(ordered)
        axes[row_index, 1].loglog(ordered, ccdf, marker='.', label=name)
    axes[row_index, 0].set(title=f'{dataset}: degree PMF', xlabel='degree', ylabel='P(k)')
    axes[row_index, 1].set(title=f'{dataset}: degree CCDF', xlabel='degree', ylabel='P(K ≥ k)')
    axes[row_index, 0].legend(); axes[row_index, 1].legend()
plt.tight_layout(); plt.show()

centrality_rows = []
for dataset, variants in graph_sets.items():
    for variant in ('exact_surface', 'conservative'):
        scores = top_centralities(variants[variant])
        for metric, ranked in scores.items():
            for rank, (node, score) in enumerate(ranked, start=1):
                centrality_rows.append({'dataset': dataset, 'graph': variant,
                    'metric': metric, 'rank': rank, 'node': node, 'score': score})
display(pd.DataFrame(centrality_rows))

In [ ]:
# Visualize one complete gold path per dataset using the supplied evidence.
fig, axes = plt.subplots(1, len(packages), figsize=(15, 5))
for axis, (dataset, package) in zip(np.atleast_1d(axes), packages.items()):
    question = package.questions('public')[0]
    path = nx.DiGraph()
    previous = 'USER QUESTION'
    path.add_node(previous, kind='question')
    for index, task in enumerate(atomic_tasks(question), start=1):
        query_node = f'Q{index}: {task["question"]}'
        answer_node = f'A{index}: {task["answer"]}'
        path.add_edge(previous, query_node, label='requires')
        path.add_edge(query_node, answer_node, label='GSW QA')
        previous = answer_node
    positions = nx.spring_layout(path, seed=CONFIG.seed)
    nx.draw_networkx(path, positions, ax=axis, node_size=800, font_size=7, arrows=True)
    nx.draw_networkx_edge_labels(path, positions,
        edge_labels=nx.get_edge_attributes(path, 'label'), ax=axis, font_size=7)
    axis.set_title(f'{dataset}: {question["question_id"]}')
    axis.axis('off')
plt.tight_layout(); plt.show()

**Interpretation.** The unreconciled giant component is
tiny because local GSWs are deliberately document-scoped. Exact
surface merging produces a much larger giant component, but that is
not evidence that write-time memory was globally connected: it is
evidence that the identity rule inserted cross-document bridges.
The audit shows why `American`, `actor`, and repeated dates are
dangerous hubs, while people such as Charles Mingus are defensible
merges. The conservative graph still changes when its rule changes,
and aliases such as USA/United States remain split. Therefore no
reconciled projection is used operationally. Entity retrieval
returns a document-local occurrence and expands only within its
originating GSW; RICR creates cross-document chains at read time.
Degree or PageRank alone cannot distinguish a real semantic hub from
a merge artifact—the role type, documents, neighborhood, and manual
audit provide that evidence. The log–log plots may look heavy-tailed,
but no power-law claim is made without a fitted comparison test.

## Question 4 — decomposition and dependency graphs (14 points)

Raw model text is appended before parsing. Validation rejects empty
plans, future or missing references, and malformed nodes. A
placeholder creates a dependency edge. Retrieval nodes form weakly
connected components that are processed in topological order. A
component may converge: a retrieval node referencing Q1 and Q2 must
receive both parent bindings before it issues its concrete query.
Deterministic comparison and intersection nodes remain distinct
because they combine retrieved values rather than search memory.

In [ ]:
if RUN_FULL:
    run_decomposition_stage(jobs, CONFIG)
else:
    print('Full decomposition skipped by RUN_FULL=False.')

decomposition_tables = []
decomposition_error_rows = []
for dataset, package, _, cache_root in jobs:
    records = read_jsonl(cache_root / 'decompositions.jsonl')
    predicted = {row['question_id']: row['predicted_decomposition']
                 for row in records if row.get('predicted_decomposition')}
    if predicted:
        decomposition_tables.append({'dataset': dataset,
            **decomposition_metrics(predicted, package.decompositions())})
        for qid, reviewed_plan in package.decompositions().items():
            predicted_plan = predicted.get(qid, [])
            reviewed_edges = set(map(tuple, validate_plan(reviewed_plan)['edges']))
            predicted_edges = set(map(tuple, validate_plan(predicted_plan)['edges']))
            if len(predicted_plan) != len(reviewed_plan):
                category = ('missing hop' if len(predicted_plan) < len(reviewed_plan)
                            else 'extra hop')
            elif predicted_edges != reviewed_edges:
                category = 'wrong dependency'
            else:
                continue
            decomposition_error_rows.append({
                'dataset': dataset, 'question_id': qid, 'category': category,
                'predicted': json.dumps(predicted_plan, ensure_ascii=False),
                'reviewed': json.dumps(reviewed_plan, ensure_ascii=False),
            })
display(pd.DataFrame(decomposition_tables))
error_frame = pd.DataFrame(decomposition_error_rows)
if not error_frame.empty:
    display(error_frame.groupby('dataset', group_keys=False).head(2))

**Error analysis.** Invalid JSON or an absent list is a
malformed-output error. A placeholder pointing to the wrong earlier
answer is a dependency error. Too few retrieval nodes is a missing
hop; an unnecessary lookup is an extra hop. A valid graph can still
ask the wrong atomic question, which is a semantic error rather than
a parser error. For a two-branch comparison, film→director and
director→death-date execute independently; “which is later?” then
compares the two dates, and the final operation maps the winning
director back to the film. It is reasoning because the corpus need
not contain that comparison as a stored QA pair.

## Question 5 — sparse retrieval baselines (12 points)

TF–IDF, BM25-QA, and BM25-entity expansion use the same normalized
evidence tasks and stable QA IDs. Entity expansion follows only the
local verb neighborhood attached to the retrieved occurrence.

In [ ]:
from panini_course import BM25Index, DualRetriever, TfidfIndex

def relevant_for_task(task, question, qa_rows):
    documents = ({task['document_id']} if task['document_id'] else
                 set(question['context_document_ids']))
    answer = normalize_text(task['answer'])
    candidates = [row for row in qa_rows
                  if row['document_id'] in documents and
                  answer in {normalize_text(value) for value in row['answer_names']}]
    if not candidates:
        return set()
    query_tokens = set(WORD.findall(task['question'].casefold())) - STOPWORDS
    def overlap(row):
        text = f"{row.get('question','')} {row.get('verb_phrase','')}"
        tokens = set(WORD.findall(text.casefold())) - STOPWORDS
        return len(query_tokens & tokens) / max(len(query_tokens | tokens), 1)
    best = sorted(candidates, key=lambda row: (-overlap(row), row['qa_uid']))[0]
    return {best['qa_uid']}

sparse_rows, sparse_disagreements = [], []
for dataset, package in packages.items():
    root, qa_rows = package.root, package.qa_pairs()
    qa_by_id = {row['qa_uid']: row for row in qa_rows}
    entity = BM25Index.load(root/'indices/entity_bm25.joblib', root/'indices/entity_ids.json')
    bm25 = BM25Index.load(root/'indices/qa_bm25.joblib', root/'indices/qa_ids.json')
    tfidf = TfidfIndex.load(root/'indices/qa_tfidf.npz',
        root/'indices/qa_tfidf_vectorizer.joblib', root/'indices/qa_ids.json')
    expansion = DualRetriever(entity_index=entity, qa_index=bm25,
        entity_rows=package.entities(), qa_rows=qa_rows)
    for question in package.questions('public'):
        group = question['type'] if dataset == '2wiki' else question['hop_count']
        for task in atomic_tasks(question):
            relevant = relevant_for_task(task, question, qa_rows)
            tfidf_hits = tfidf.search(task['question'], 15)
            bm25_hits = bm25.search(task['question'], 15)
            entity_hits = entity.search(task['question'], 20)
            expanded = []
            for hit in entity_hits:
                expanded.extend(expansion.qa_ids_by_entity.get(hit.item_id, ()))
            rankings = {
                'tfidf': [hit.item_id for hit in tfidf_hits],
                'bm25_qa': [hit.item_id for hit in bm25_hits],
                'bm25_entity_expansion': list(dict.fromkeys(expanded))[:15],
            }
            if (rankings['tfidf'][:1] != rankings['bm25_qa'][:1]
                    and sum(row['dataset'] == dataset for row in sparse_disagreements) < 2):
                query_tokens = set(WORD.findall(task['question'].casefold()))
                for method, hits in [('tfidf', tfidf_hits[:5]), ('bm25_qa', bm25_hits[:5])]:
                    for hit in hits:
                        qa = qa_by_id[hit.item_id]
                        matched = sorted(query_tokens & set(WORD.findall(qa['search_text'].casefold())))
                        sparse_disagreements.append({
                            'dataset': dataset, 'query': task['question'],
                            'method': method, 'rank': hit.rank, 'score': hit.score,
                            'stored_question': qa['question'],
                            'answer': '; '.join(qa['answer_names']),
                            'matched_terms': ', '.join(matched),
                        })
            for method, ranked in rankings.items():
                for k in (1, 5, 10, 15):
                    sparse_rows.append({'dataset': dataset, 'group': group,
                        'method': method, 'k': k,
                        **evaluate_ranked_ids(ranked, relevant, k)})
sparse_results = pd.DataFrame(sparse_rows)
display(sparse_results.groupby(['dataset','group','method','k'], as_index=False)
        [['recall','reciprocal_rank']].mean())
display(pd.DataFrame(sparse_disagreements))

**Interpretation.** TF–IDF strongly rewards rare exact
terms but does not saturate repeated term frequency. BM25 adds term
saturation and document-length normalization, which can move a short
focused QA above a long record containing the same words. Entity
expansion solves a different problem: a named surface can retrieve
a local node even when the attached QA paraphrases the query. It can
also add irrelevant sibling questions, so it is a recall route, not
a final ranker. Disagreements should be explained using the displayed
terms, lengths, document frequencies, and source entity—not by
saying one method is simply “more semantic.”

## Questions 6–7 — dense, RRF, dual retrieval, and reranking (24 points)

All query vectors reached by the required deterministic runs are
supplied, so Stage B loads no embedding model. It loads the 4-bit
Qwen3-Reranker-8B alone with batch size 1 and 256-token inputs; if a
particular T4 raises an OOM, it records and uses the official 4B
fallback. It warms the exact gold atomic-query rankings and then
runs every predicted plan.
Each query cache stores BM25, dense, RRF, reranker-only,
retrieval-only, and 0.5/0.5 hybrid rankings over the same dual pool.
The fixed candidate pool makes the Question 7 comparison controlled.

In [ ]:
if RUN_FULL:
    run_neural_retrieval_stage_low_memory(
        jobs, CONFIG, run_ablations=RUN_ABLATIONS)
else:
    print('Full neural retrieval skipped by RUN_FULL=False.')

reranking_rows, disagreement_rows = [], []
for dataset, package, _, cache_root in jobs:
    cache = {row['query_key']: row for row in read_jsonl(cache_root/'retrieval_cache.jsonl')}
    qa_rows = package.qa_pairs()
    for question in package.questions('public'):
        for task in atomic_tasks(question):
            record = cache.get(normalize_text(task['question']))
            if not record:
                continue
            relevant = relevant_for_task(task, question, qa_rows)
            ranks = {}
            for method in ('bm25','dense','rrf','retrieval_only','reranker_only','dual_hybrid'):
                ids = [row['qa_uid'] for row in record['rankings'][method]]
                metric = evaluate_ranked_ids(ids, relevant, 15)
                ranks[method] = metric['reciprocal_rank']
                reranking_rows.append({'dataset': dataset, 'method': method, **metric})
            disagreement_rows.append({'dataset': dataset, 'query': task['question'], **ranks})
reranking_table = pd.DataFrame(reranking_rows)
if not reranking_table.empty:
    display(reranking_table.groupby(['dataset','method'], as_index=False).mean(numeric_only=True))
    disagreements = pd.DataFrame(disagreement_rows)
    display(disagreements.sort_values('reranker_only', ascending=False).head(5))
    display(disagreements.sort_values('reranker_only', ascending=True).head(5))
    disagreements['rerank_delta'] = (
        disagreements.reranker_only - disagreements.retrieval_only
    )
    trace_queries = [
        disagreements.sort_values('rerank_delta', ascending=False).iloc[0],
        disagreements.sort_values('rerank_delta', ascending=True).iloc[0],
    ]
    annotated = []
    for selected in trace_queries:
        cache_root = CACHE_BASE / selected.dataset
        record = {row['query_key']: row for row in
                  read_jsonl(cache_root/'retrieval_cache.jsonl')}[
                      normalize_text(selected['query'])]
        for method in ('retrieval_only', 'reranker_only'):
            for rank, candidate in enumerate(record['rankings'][method][:5], start=1):
                annotated.append({'dataset': selected.dataset,
                    'query': selected['query'], 'delta': selected.rerank_delta,
                    'method': method, 'rank': rank,
                    'qa_uid': candidate['qa_uid'], 'answer': candidate['answer'],
                    'score': candidate['score'],
                    'reranker_score': candidate.get('reranker_score'),
                    'retrieval_rank': candidate.get('retrieval_rank'),
                    'routes': candidate.get('routes')})
    display(pd.DataFrame(annotated))

    helped = disagreements[
        disagreements.dual_hybrid > disagreements[['dense','retrieval_only']].max(axis=1)
    ]
    if not helped.empty:
        selected = helped.iloc[0]
        record = {row['query_key']: row for row in
                  read_jsonl((CACHE_BASE/selected.dataset)/'retrieval_cache.jsonl')}[
                      normalize_text(selected['query'])]
        display(pd.DataFrame(record['rankings']['dual_hybrid'][:15])[
            ['qa_uid','answer','score','routes']])

In [ ]:
# Manual FAISS inner-product consistency for five supplied fixed queries.
from panini_course import DenseIndex, QueryEmbeddingStore
consistency = []
for dataset, package in packages.items():
    root = package.root
    dense = DenseIndex.load(root/'indices/qa_qwen3_8b_ip.faiss', root/'indices/qa_ids.json')
    store = QueryEmbeddingStore.load(root/'embeddings/query_embeddings.npy',
        root/'embeddings/query_ids.json', root/'embeddings/queries.jsonl')
    matrix = np.load(root/'embeddings/qa_embeddings.npy', mmap_mode='r')
    qa_ids = json.loads((root/'embeddings/qa_ids.json').read_text())
    query_rows = read_jsonl(root/'embeddings/queries.jsonl')[:5]
    for query_row in query_rows:
        query_text = query_row['text']
        vector = store.get(query_text)
        manual = qa_ids[int(np.argmax(matrix @ vector))]
        faiss_top = dense.search(vector, 1)[0].item_id
        consistency.append({'dataset': dataset, 'query': query_text,
                            'manual_top': manual, 'faiss_top': faiss_top,
                            'match': manual == faiss_top})
display(pd.DataFrame(consistency))
assert all(row['match'] for row in consistency)

**Interpretation.** Dense QA retrieval helps when the
atomic query paraphrases stored text, while entity expansion helps
when a named node is a stronger doorway than the QA wording. Their
union therefore changes the candidate set, not merely the score.
RRF makes lexical and dense ranks comparable without pretending
their raw scores share a scale. Reranking can improve a relevant
candidate by reading query and QA jointly, but it can also demote a
terse correct relation in favor of a fluent topical distractor. The
controlled table answers “does it help?” empirically: compare first
relevant ranks within the identical candidate pool and inspect the
reranker probability alongside the retrieval prior.

## Question 8 — complete RICR implementation (22 points)

The reference solution executes connected retrieval DAGs, not
independent linear branches. It topologically processes a component;
a multi-parent node ranks Cartesian products by harmonic mean before
substituting all parents. Intermediate hops keep the best state per
namespaced answer entity, whereas the final hop keeps QA records
directly. Evidence is the deduplicated union from every surviving
final beam. These are the exact research-code semantics at a smaller
corpus/model scale.

In [ ]:
toy_plan = [
    {'question': 'Who founded lab A?', 'requires_retrieval': True},
    {'question': 'Who founded lab B?', 'requires_retrieval': True},
    {'question': 'Who published first, <ENTITY_Q1> or <ENTITY_Q2>?',
     'requires_retrieval': True},
]
toy = {
    'Who founded lab A?': [
        {'qa_uid':'q1','answer_names':['Ada'],'answer_ids':['d1::e1'],
         'question':'founder A','document_id':'d1','score':0.90},
        {'qa_uid':'q2','answer_names':['Alan'],'answer_ids':['d2::e1'],
         'question':'founder A','document_id':'d2','score':0.60}],
    'Who founded lab B?': [
        {'qa_uid':'q3','answer_names':['Grace'],'answer_ids':['d3::e1'],
         'question':'founder B','document_id':'d3','score':0.80},
        {'qa_uid':'q4','answer_names':['Katherine'],'answer_ids':['d4::e1'],
         'question':'founder B','document_id':'d4','score':0.70}],
    'Who published first, Ada or Grace?': [
        {'qa_uid':'q5','answer_names':['Ada'],'answer_ids':['d1::e1'],
         'question':'comparison','document_id':'d5','score':0.95}],
    'Who published first, Ada or Katherine?': [
        {'qa_uid':'q6','answer_names':['Ada'],'answer_ids':['d1::e1'],
         'question':'comparison','document_id':'d6','score':0.75}],
}
toy_result = execute_panini_plan(
    toy_plan, lambda query, k: toy[query][:k],
    replace(CONFIG, beam_width=2, candidates_per_hop=2),
    original_question='Who published first?')
display(pd.DataFrame(toy_result['component_traces'][0]['steps'][-1]['kept']))
print(json.dumps(toy_result['component_traces'], indent=2))
print('all-final-beam evidence:', [row['qa_uid'] for row in toy_result['evidence']])

**Hand calculation.** The normalized root scores are Ada `.95`,
Alan `.80`, Grace `.90`, and Katherine `.85`. The four parent
products have harmonic scores `.924`, `.897`, `.847`, and `.824` in
that order, so `B=2` sends only `Ada + Grace` and `Ada + Katherine`
to the converging retrieval node. Both final QA records answer Ada,
but final-hop selection does not entity-deduplicate them. Their
parent evidence also remains in the context because evidence is
collected from both final beams, not just the best one.

## Question 9 — controlled RICR ablations (14 points)

The fixed-seed subset contains 20 development questions per
dataset. Before seeing results, the directional predictions are:
narrower beams and `k=5` reduce latency but lower chain recovery;
disabling intermediate entity grouping lowers substitution diversity;
removing the multi-parent threshold increases joint queries; last-hop
scoring is less stable because it forgets early weak links; BM25
loses paraphrases; dense-only loses entity-doorway candidates; RRF
is competitive but lacks the cross-encoder's joint judgment.

In [ ]:
prediction_table = pd.DataFrame([
    {'configuration':'beam_1', 'prediction':'lower chain recovery and latency'},
    {'configuration':'beam_3', 'prediction':'between beam 1 and beam 5'},
    {'configuration':'k_5', 'prediction':'lower recall and reranking cost'},
    {'configuration':'unique_off', 'prediction':'fewer distinct substitutions'},
    {'configuration':'parent_threshold_off', 'prediction':'more joint queries and latency'},
    {'configuration':'last_hop', 'prediction':'unstable; ignores weak early evidence'},
    {'configuration':'bm25', 'prediction':'loses paraphrased evidence'},
    {'configuration':'dense', 'prediction':'loses entity-doorway candidates'},
    {'configuration':'rrf', 'prediction':'better coverage than one route but no cross-encoder'},
])
display(prediction_table)

ablation_frames = []
for dataset, _, _, cache_root in jobs:
    rows = read_jsonl(cache_root/'ablation_answers.jsonl')
    if rows:
        frame = pd.DataFrame(rows)
        summary = frame.groupby('configuration', as_index=False).agg(
            questions=('question_id','count'),
            supporting_qa_recall=('supporting_qa_recall','mean'),
            complete_chain_recovery=('complete_chain_recovery','mean'),
            EM=('exact_match','mean'), F1=('token_f1','mean'),
            latency=('retrieval_seconds','mean'),
            evidence_count=('evidence_count','mean'))
        summary.insert(0, 'dataset', dataset)
        ablation_frames.append(summary)
if ablation_frames:
    ablation_table = pd.concat(ablation_frames, ignore_index=True)
    display(ablation_table)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for dataset, rows in ablation_table.groupby('dataset'):
        axes[0].scatter(rows.latency, rows.F1, label=dataset)
        axes[1].scatter(rows.evidence_count, rows.F1, label=dataset)
    axes[0].set(xlabel='mean retrieval seconds', ylabel='F1', title='Accuracy vs latency')
    axes[1].set(xlabel='mean evidence count', ylabel='F1', title='Accuracy vs evidence')
    axes[0].legend(); axes[1].legend(); plt.tight_layout(); plt.show()
else:
    print('Ablation answers appear after stages B and C finish.')

**Recommendation.** Deploy the default only if its
measured F1/latency point dominates the cheaper alternatives. Beam
width buys protection against an early retrieval error, but returns
diminish once the correct answer is already retained. Candidate
count has a similar cost because every added item reaches the
reranker. Unique-answer pruning is retained because duplicated
surface forms consume capacity without creating a new substitution.
Geometric-mean scoring is retained because a chain should not be
rescued by one strong final hop after weak earlier evidence. The
retrieval choice is made from the measured table rather than model
size: dual retrieval is justified only when its additional
supporting-QA recall survives reranking and improves complete-chain
recovery enough to offset latency.

## Questions 10–11 — frozen 2Wiki run and MuSiQue transfer (22 points)

Stage C unloads retrieval models, loads only the 4-bit Qwen3-4B
answerer, and supplies deduplicated evidence from all surviving
RICR chains, including answer role/state strings. It uses the same
four-message one-shot `Thought:`/`Answer:` prompt as the research
evaluator and does not add an N/A instruction for these answerable
splits. It never receives source documents, graph neighbors, or
labels. The configuration is identical for both datasets. Outputs
are split into 80-row development and 20-row held-out files.

In [ ]:
if RUN_FULL:
    run_answer_stage(jobs, OUTPUT_ROOT, CONFIG, run_ablations=RUN_ABLATIONS)
else:
    print('Full answer generation skipped by RUN_FULL=False.')

final_tables = []
for dataset, package in packages.items():
    records = read_jsonl(OUTPUT_ROOT/'results'/f'{dataset}_dev.jsonl')
    if not records:
        continue
    question_by_id = {row['question_id']: row for row in package.questions('public')}
    group_field = 'type' if dataset == '2wiki' else 'hop_count'
    enriched = [{**row, group_field: question_by_id[row['question_id']][group_field]}
                for row in records]
    summary = pd.DataFrame(result_summary(enriched, group_field))
    summary.insert(0, 'dataset', dataset)
    final_tables.append(summary)
    successful = next((row for row in enriched
                       if row.get('complete_chain_recovery') == 1
                       and row.get('exact_match') == 1), None)
    failed = next((row for row in enriched if row.get('exact_match') == 0), None)
    trace_by_id = {row['question_id']: row for row in
                   read_jsonl((CACHE_BASE/dataset)/'traces.jsonl')}
    for label, selected in [('successful', successful), ('failed', failed)]:
        if selected is None:
            continue
        if not selected.get('decomposition_valid'):
            first_error = 'decomposition'
        elif selected.get('supporting_qa_recall', 0) == 0:
            first_error = 'first-hop/candidate retrieval'
        elif selected.get('complete_chain_recovery', 0) == 0:
            first_error = 'later-hop substitution or global pruning'
        elif selected.get('exact_match', 0) == 0:
            first_error = 'answer generation after complete evidence'
        else:
            first_error = 'none'
        print(dataset, label, 'trace:', selected['question_id'],
              'first irreversible error:', first_error)
        print(json.dumps(trace_by_id.get(selected['question_id'], {}),
                         ensure_ascii=False, indent=2)[:8000])
if final_tables:
    final_table = pd.concat(final_tables, ignore_index=True)
    display(final_table)
    if 'hop_count' in final_table:
        musique = final_table[final_table.dataset == 'musique']
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        axes[0].plot(musique.hop_count, musique.complete_chain_recovery, marker='o')
        axes[1].plot(musique.hop_count, musique.F1, marker='o')
        axes[0].set(xlabel='hop count', ylabel='complete-chain recovery')
        axes[1].set(xlabel='hop count', ylabel='F1')
        plt.tight_layout(); plt.show()

**Transfer interpretation.** Complete-chain recovery is
conjunctive: every required hop must survive, so one additional hop
adds another opportunity for irreversible failure. Answer F1 is not
strictly conjunctive. The answer model may recover a short answer
from partial or redundant evidence, or receive the final fact even
when an intermediate gold QA was missed. Therefore chain recovery
can fall faster than F1. Hop count is not the only dataset change;
wording, entity distribution, and GSW coverage also differ. The
saved traces attribute transfer loss by counting invalid plans,
missing first-hop gold answers, successful first hops followed by
missing later answers, and correct evidence followed by wrong
generation. That separation is stronger evidence than attributing
every MuSiQue loss to length.

## Question 12 — reproducibility and handoff (12 points)

The last cell verifies the required file counts, records the runtime
and model configuration, and writes exact resume instructions. A
cache is complete only when its stable question IDs match the input
IDs; file existence alone is not sufficient.

In [ ]:
required = {
    OUTPUT_ROOT/'results/2wiki_dev.jsonl': 80,
    OUTPUT_ROOT/'predictions/2wiki_heldout.jsonl': 20,
    OUTPUT_ROOT/'results/musique_dev.jsonl': 80,
    OUTPUT_ROOT/'predictions/musique_heldout.jsonl': 20,
}
if QUESTION_SLICE == slice(None) and RUN_FULL:
    for path, expected in required.items():
        actual = len(read_jsonl(path))
        assert actual == expected, f'{path}: expected {expected}, found {actual}'
write_environment(OUTPUT_ROOT/'environment.txt', CONFIG)
runme = chr(10).join([
    '# PANINI full run', '',
    '1. Open this notebook in a fresh Colab GPU runtime.',
    '2. Leave `QUESTION_SLICE = slice(None)` and run all cells.',
    '3. If the runtime stops, reconnect and run all again; JSONL caches skip completed IDs.',
    f'4. Final files are under `{OUTPUT_ROOT}`.',
    f'5. Configuration: `{asdict(CONFIG)}`.',
    '',
])
(OUTPUT_ROOT/'RUNME.md').write_text(runme, encoding='utf-8')
print({str(path): len(read_jsonl(path)) for path in required})
print('environment:', OUTPUT_ROOT/'environment.txt')
print('run guide:', OUTPUT_ROOT/'RUNME.md')

**System story.** A user question first becomes a validated
dependency graph. Each retrieval component is topologically
executed; converging nodes combine parent beams before substituting
all answers. BM25 entity expansion and dense QA search create a
union candidate pool, and the reranker orders it. RICR groups
intermediate answer entities but retains final QA alternatives.
Evidence from all surviving final beams is deduplicated by stable QA
ID before it becomes the answer model's only context. This design
preserves provenance and makes
failures inspectable. Its main protection is beam diversity: a
plausible but wrong first hop does not immediately destroy the
correct path. Its main failure mode is earlier than generation: if
decomposition omits a hop, or the correct answer never enters the
candidate pool, later reranking and answer generation cannot invent
grounded evidence. The raw plan, per-query rankings, beam trace, and
final answer are cached separately so that the first irreversible
error can be located rather than guessed.